## Check Notebook Kernel

Verify that the notebook is using the project virtual environment so package installs and imports come from the right Python.


In [522]:
import sys
print(sys.executable)

c:\Users\achur\desktop\AISE_CLASS_CLONES\march-madness-bracket-simulator\.venv\Scripts\python.exe


## Confirm Core Library Versions

Import the main analysis libraries and print their versions to confirm the environment is ready for data work.


In [523]:
import pandas as pd
import numpy as np
import sklearn

print(pd.__version__)
print(np.__version__)
print(sklearn.__version__)

2.3.3
2.4.3
1.8.0


## Inspect Raw Data Files

Point to the raw data folder and list available CSV files so we know what Kaggle data is present in the project.


In [524]:
from pathlib import Path

data_dir = Path("../data/raw")
sorted([p.name for p in data_dir.glob("*.csv")])[:20]


['Cities.csv',
 'Conferences.csv',
 'MConferenceTourneyGames.csv',
 'MGameCities.csv',
 'MMasseyOrdinals.csv',
 'MNCAATourneyCompactResults.csv',
 'MNCAATourneyDetailedResults.csv',
 'MNCAATourneySeedRoundSlots.csv',
 'MNCAATourneySeeds.csv',
 'MNCAATourneySlots.csv',
 'MRegularSeasonCompactResults.csv',
 'MRegularSeasonDetailedResults.csv',
 'MSeasons.csv',
 'MSecondaryTourneyCompactResults.csv',
 'MSecondaryTourneyTeams.csv',
 'MTeamCoaches.csv',
 'MTeamConferences.csv',
 'MTeamSpellings.csv',
 'MTeams.csv',
 'SampleSubmissionStage1.csv']

## Load Core Men's NCAA Datasets

This cell loads the main datasets needed for the project:
- teams
- regular season results
- tournament results
- seeds
- bracket slots


In [525]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/raw")

m_teams = pd.read_csv(data_dir / "MTeams.csv")
m_reg = pd.read_csv(data_dir / "MRegularSeasonCompactResults.csv")
m_tourney = pd.read_csv(data_dir / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(data_dir / "MNCAATourneySeeds.csv")
m_slots = pd.read_csv(data_dir / "MNCAATourneySlots.csv")




## Inspect Shapes and Columns

Print the size and column layout of each core dataset to understand the keys and what each file contains.


In [526]:
datasets = {
    "teams": m_teams,
    "regular_season": m_reg,
    "tourney": m_tourney,
    "seeds": m_seeds,
    "slots": m_slots,
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())

--- teams ---
shape: (381, 4)
columns: ['TeamID', 'TeamName', 'FirstD1Season', 'LastD1Season']


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


--- regular_season ---
shape: (198079, 8)
columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0


--- tourney ---
shape: (2585, 8)
columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


--- seeds ---
shape: (2626, 3)
columns: ['Season', 'Seed', 'TeamID']


,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374


--- slots ---
shape: (2586, 4)
columns: ['Season', 'Slot', 'StrongSeed', 'WeakSeed']


,Season,Slot,StrongSeed,WeakSeed
0,1985,R1W1,W01,W16
1,1985,R1W2,W02,W15
2,1985,R1W3,W03,W14
3,1985,R1W4,W04,W13
4,1985,R1W5,W05,W12


## Preview Bracket Slot Structure

Look at the first tournament slot rows to see how first-round seed matchups are encoded.


In [527]:
m_slots.head(15)


,Season,Slot,StrongSeed,WeakSeed
0,1985,R1W1,W01,W16
1,1985,R1W2,W02,W15
2,1985,R1W3,W03,W14
3,1985,R1W4,W04,W13
4,1985,R1W5,W05,W12
5,1985,R1W6,W06,W11
6,1985,R1W7,W07,W10
7,1985,R1W8,W08,W09
8,1985,R1X1,X01,X16
9,1985,R1X2,X02,X15


## Check Slot Season Coverage

Confirm the season range available in the tournament slot table.


In [528]:
m_slots["Season"].min(), m_slots["Season"].max()


(np.int64(1985), np.int64(2025))

## Inspect Latest Slot Layout

Preview the most recent season's slot rows to see how the current bracket structure is represented.


In [529]:
m_slots[m_slots["Season"] == m_slots["Season"].max()].head(20)


,Season,Slot,StrongSeed,WeakSeed
2519,2025,R1W1,W01,W16
2520,2025,R1W2,W02,W15
2521,2025,R1W3,W03,W14
2522,2025,R1W4,W04,W13
2523,2025,R1W5,W05,W12
2524,2025,R1W6,W06,W11
2525,2025,R1W7,W07,W10
2526,2025,R1W8,W08,W09
2527,2025,R1X1,X01,X16
2528,2025,R1X2,X02,X15


## Inspect Historical Bracket Progression

Use the 1985 slot rows to understand how later rounds depend on earlier slot winners.


In [530]:
m_slots[m_slots["Season"] == 1985].tail(20)


,Season,Slot,StrongSeed,WeakSeed
43,1985,R2Y4,R1Y4,R1Y5
44,1985,R2Z1,R1Z1,R1Z8
45,1985,R2Z2,R1Z2,R1Z7
46,1985,R2Z3,R1Z3,R1Z6
47,1985,R2Z4,R1Z4,R1Z5
48,1985,R3W1,R2W1,R2W4
49,1985,R3W2,R2W2,R2W3
50,1985,R3X1,R2X1,R2X4
51,1985,R3X2,R2X2,R2X3
52,1985,R3Y1,R2Y1,R2Y4


## Test Reusable Data Loaders

Load the same core datasets through the project's Python loader module to confirm reusable code works outside the notebook.


In [531]:
from march_madness_bracket_simulator.data_loader import load_core_march_madness_data

data = load_core_march_madness_data()

for name, df in data.items():
    print(name, df.shape)


teams (381, 4)
regular_season (198079, 8)
tourney (2585, 8)
seeds (2626, 3)
slots (2586, 4)
bracket_2026 (64, 3)


## Build First Team-Season Feature Table

Create a basic feature table from regular-season results with wins, losses, scoring, and margin statistics.


In [532]:
from march_madness_bracket_simulator.feature_engineering import build_team_season_features

team_features = build_team_season_features(data["regular_season"])
team_features.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000


## Merge Team Names and Seeds

Attach readable team names and tournament seeds to the team-season features so the table is easier to interpret.


In [533]:
team_features_named = team_features.merge(
    data["teams"][["TeamID", "TeamName"]],
    on="TeamID",
    how="left"
)

team_features_named = team_features_named.merge(
    data["seeds"],
    on=["Season", "TeamID"],
    how="left"
)

team_features_named.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667,Air Force,NaN
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478,Akron,NaN
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama,X07
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667,Alabama St,NaN
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000,Alcorn St,NaN


## Review One Historical Season

Sort one older season by scoring margin to see whether the feature table surfaces strong teams sensibly.


In [534]:
team_features_named[team_features_named["Season"] == 1985].sort_values(
    "avg_scoring_margin", ascending=False
).head(15)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
80,1985,1207,25.0,2.0,27.0,0.925926,75.740741,60.074074,15.666667,Georgetown,W01
174,1985,1328,25.0,5.0,30.0,0.833333,89.833333,76.533333,13.300000,Oklahoma,Y01
118,1985,1256,26.0,2.0,28.0,0.928571,77.500000,64.821429,12.678571,Louisiana Tech,Y05
261,1985,1439,19.0,8.0,27.0,0.703704,81.148148,69.185185,11.962963,Virginia Tech,W09
217,1985,1385,27.0,3.0,30.0,0.900000,76.133333,64.400000,11.733333,St John's,X01
60,1985,1181,22.0,7.0,29.0,0.758621,79.241379,67.896552,11.344828,Duke,Y03
152,1985,1298,23.0,5.0,28.0,0.821429,78.214286,67.321429,10.892857,Navy,Z13
98,1985,1228,23.0,8.0,31.0,0.741935,68.225806,57.354839,10.870968,Illinois,W03
136,1985,1276,25.0,3.0,28.0,0.892857,79.750000,69.107143,10.642857,Michigan,Z01
103,1985,1234,20.0,10.0,30.0,0.666667,69.733333,59.266667,10.466667,Iowa,X08


## Step 1: Attach Team Names

Merge the feature table with the team lookup table using `TeamID` so rows are readable by school name.


In [535]:
team_features_named = team_features.merge(
    data["teams"][["TeamID", "TeamName"]],
    on="TeamID",
    how="left"
)

## Preview Features With Team Names

Check the feature table after adding team names to make sure the merge behaved correctly.


In [536]:
team_features_named.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667,Air Force
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478,Akron
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667,Alabama St
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000,Alcorn St


## Step 2: Attach Tournament Seeds

Merge tournament seeds onto the feature table using `Season` and `TeamID` because seeds change by season.


In [537]:
team_features_named = team_features_named.merge(
    data["seeds"],
    on=["Season", "TeamID"],
    how="left"
)

## Preview Tournament Teams Only

Filter to rows with a non-null seed to verify that tournament teams now carry bracket seed information.


In [538]:
team_features_named[team_features_named["Seed"].notna()].head(10)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama,X07
8,1985,1112,18.0,9.0,27.0,0.666667,66.518519,59.333333,7.185185,Arizona,X10
11,1985,1116,21.0,12.0,33.0,0.636364,65.333333,61.696970,3.636364,Arkansas,X09
14,1985,1120,18.0,11.0,29.0,0.620690,70.344828,66.655172,3.689655,Auburn,Z11
21,1985,1130,16.0,10.0,26.0,0.615385,74.884615,69.615385,5.269231,Boston College,Y11
53,1985,1173,19.0,9.0,28.0,0.678571,69.535714,64.607143,4.928571,Dayton,Z09
56,1985,1177,18.0,9.0,27.0,0.666667,71.666667,64.000000,7.666667,DePaul,W10
60,1985,1181,22.0,7.0,29.0,0.758621,79.241379,67.896552,11.344828,Duke,Y03
69,1985,1192,19.0,9.0,28.0,0.678571,67.892857,64.714286,3.178571,F Dickinson,Z16
80,1985,1207,25.0,2.0,27.0,0.925926,75.740741,60.074074,15.666667,Georgetown,W01


## Look at a Recent Season

Sort a recent season by scoring margin to get a more relevant feel for current-era team strength.


In [539]:
team_features_named[team_features_named["Season"] == 2025].sort_values(
    "avg_scoring_margin", ascending=False
).head(20)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
13098,2025,1181,31.0,3.0,34.0,0.911765,82.705882,61.911765,20.794118,Duke,W01
13128,2025,1211,25.0,8.0,33.0,0.757576,86.636364,69.636364,17.000000,Gonzaga,X08
13113,2025,1196,30.0,4.0,34.0,0.882353,85.411765,69.235294,16.176471,Florida,Z01
13137,2025,1222,30.0,4.0,34.0,0.882353,74.205882,58.470588,15.735294,Houston,X01
13378,2025,1471,28.0,4.0,32.0,0.875000,77.937500,62.843750,15.093750,UC San Diego,Y12
13183,2025,1268,25.0,8.0,33.0,0.757576,81.666667,67.000000,14.666667,Maryland,Z04
13041,2025,1120,28.0,5.0,33.0,0.848485,83.848485,69.606061,14.242424,Auburn,Y01
13342,2025,1433,27.0,6.0,33.0,0.818182,76.333333,62.515152,13.818182,VCU,W11
13313,2025,1403,25.0,8.0,33.0,0.757576,80.909091,67.575758,13.333333,Texas Tech,Z03
13295,2025,1385,30.0,4.0,34.0,0.882353,78.705882,65.882353,12.823529,St John's,Z02


## Look at Recent Seeded Teams

Focus on seeded teams in a recent season so the table is closer to the tournament use case.


In [540]:
team_features_named[
    (team_features_named["Season"] == 2025) &
    (team_features_named["Seed"].notna())
].sort_values("avg_scoring_margin", ascending=False).head(20)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
13098,2025,1181,31.0,3.0,34.0,0.911765,82.705882,61.911765,20.794118,Duke,W01
13128,2025,1211,25.0,8.0,33.0,0.757576,86.636364,69.636364,17.000000,Gonzaga,X08
13113,2025,1196,30.0,4.0,34.0,0.882353,85.411765,69.235294,16.176471,Florida,Z01
13137,2025,1222,30.0,4.0,34.0,0.882353,74.205882,58.470588,15.735294,Houston,X01
13378,2025,1471,28.0,4.0,32.0,0.875000,77.937500,62.843750,15.093750,UC San Diego,Y12
13183,2025,1268,25.0,8.0,33.0,0.757576,81.666667,67.000000,14.666667,Maryland,Z04
13041,2025,1120,28.0,5.0,33.0,0.848485,83.848485,69.606061,14.242424,Auburn,Y01
13342,2025,1433,27.0,6.0,33.0,0.818182,76.333333,62.515152,13.818182,VCU,W11
13313,2025,1403,25.0,8.0,33.0,0.757576,80.909091,67.575758,13.333333,Texas Tech,Z03
13295,2025,1385,30.0,4.0,34.0,0.882353,78.705882,65.882353,12.823529,St John's,Z02


## Load the 2026 Bracket File

Load the manually created bracket CSV so it can be matched against Kaggle team names and IDs.


In [541]:
bracket_2026 = pd.read_csv(data_dir / "bracket_2026.csv")
m_teams = pd.read_csv(data_dir / "MTeams.csv")

bracket_2026.head()

,region,seed,team
0,East,1,Duke
1,East,16,Siena
2,East,8,Ole Miss
3,East,9,TCU
4,East,5,St. John's


## First Pass Team Matching

Try a direct merge from bracket team names to Kaggle team names to see which teams map cleanly.


In [542]:
team_lookup = m_teams[["TeamID", "TeamName"]].copy()

merged = bracket_2026.merge(
    team_lookup,
    left_on="team",
    right_on="TeamName",
    how="left"
)

merged[["region", "seed", "team", "TeamID", "TeamName"]]


,region,seed,team,TeamID,TeamName
0,East,1,Duke,1181.0,Duke
1,East,16,Siena,1373.0,Siena
2,East,8,Ole Miss,NaN,NaN
3,East,9,TCU,1395.0,TCU
4,East,5,St. John's,NaN,NaN
...,...,...,...,...,...
59,Midwest,14,Wright St.,NaN,NaN
60,Midwest,7,Kentucky,1246.0,Kentucky
61,Midwest,10,Santa Clara,1365.0,Santa Clara
62,Midwest,2,Iowa St.,NaN,NaN


## Find Unmatched Bracket Teams

List bracket team names that failed to match so we can normalize them before modeling.


In [543]:
merged[merged["TeamID"].isna()][["region", "seed", "team"]]


,region,seed,team
2,East,8,Ole Miss
4,East,5,St. John's
10,East,3,Michigan St.
11,East,14,North Dakota St.
14,East,2,UConn
17,South,16,PVAMU
21,South,12,McNeese
28,South,7,Saint Mary's
33,West,16,Long Island
35,West,9,Utah St.


## Create a Manual Name Mapping

Define known name corrections where the bracket display names do not exactly match Kaggle team names.


In [544]:
name_map = {
    "Ole Miss": "Mississippi",
    "St. John's": "St John's",
    "Michigan St.": "Michigan St",
    "North Dakota St.": "North Dakota St",
    "UConn": "Connecticut",
    "McNeese": "McNeese St",
    "Saint Mary's": "Saint Mary's CA",
    "Long Island": "LIU Brooklyn",   # may need checking
    "Utah St.": "Utah St",
    "Kennesaw St.": "Kennesaw",
    "Miami (FL)": "Miami FL",
    "Queens (N.C.)": "Queens NC",
    "Saint Louis": "St Louis",
    "Wright St.": "Wright St",
    "Iowa St.": "Iowa St",
    "Tennessee St.": "Tennessee St",
    "PVAMU": "Prairie View",

    
}


## Apply Name Cleanup

Create a cleaned team-name column that will be used for a second, more accurate team-ID merge.


In [545]:
bracket_2026["team_clean"] = bracket_2026["team"].replace(name_map)


## Step 1: make a copy of the bracket table

In [546]:
bracket_clean = bracket_2026.copy()


### Why:

so you don’t keep mutating the original
it gives you a clean working version

## Step 2: add a play-in flag

In [547]:
bracket_clean["is_play_in"] = bracket_clean["team"].str.contains("/", regex=False)


### What this means:

if the team name has /, it is a play-in row
those are not normal one-team bracket rows yet

In [548]:
bracket_clean[["team", "is_play_in"]].head(20)


,team,is_play_in
0,Duke,False
1,Siena,False
2,Ole Miss,False
3,TCU,False
4,St. John's,False
5,Northern Iowa,False
6,Kansas,False
7,Cal Baptist,False
8,Louisville,False
9,South Florida,False


## Step 3: build the cleaned-name column

In [549]:
bracket_clean["team_clean"] = bracket_clean["team"].replace(name_map)


### Why:

- keep original team
- create a cleaned version for matching

In [550]:
bracket_clean[["team", "team_clean", "is_play_in"]].head(25)


,team,team_clean,is_play_in
0,Duke,Duke,False
1,Siena,Siena,False
2,Ole Miss,Mississippi,False
3,TCU,TCU,False
4,St. John's,St John's,False
5,Northern Iowa,Northern Iowa,False
6,Kansas,Kansas,False
7,Cal Baptist,Cal Baptist,False
8,Louisville,Louisville,False
9,South Florida,South Florida,False


## What I'm checking:

- normal names stay normal
- mismatched names change to Kaggle-style names
- play-in rows still exist and are visible

## Step 4: merge again using team_clean

In [551]:
merged_clean = bracket_clean.merge(
    team_lookup,
    left_on="team_clean",
    right_on="TeamName",
    how="left"
)


### What this means:

- use the cleaned names instead of the original bracket display names
- try the team match again

## Step 5: inspect the result

In [552]:
merged_clean[["region", "seed", "team", "team_clean", "is_play_in", "TeamID", "TeamName"]].head(25)


,region,seed,team,team_clean,is_play_in,TeamID,TeamName
0,East,1,Duke,Duke,False,1181.0,Duke
1,East,16,Siena,Siena,False,1373.0,Siena
2,East,8,Ole Miss,Mississippi,False,1279.0,Mississippi
3,East,9,TCU,TCU,False,1395.0,TCU
4,East,5,St. John's,St John's,False,1385.0,St John's
5,East,12,Northern Iowa,Northern Iowa,False,1320.0,Northern Iowa
6,East,4,Kansas,Kansas,False,1242.0,Kansas
7,East,13,Cal Baptist,Cal Baptist,False,1465.0,Cal Baptist
8,East,6,Louisville,Louisville,False,1257.0,Louisville
9,East,11,South Florida,South Florida,False,1378.0,South Florida


### What I'm checking:

- the names I cleaned earlier should now have TeamID
- play-in rows will probably still be unmatched
- a few tricky school names may still fail

## Step 6: see what is still unmatched

In [553]:
merged_clean[merged_clean["TeamID"].isna()][["region", "seed", "team", "team_clean", "is_play_in"]]


,region,seed,team,team_clean,is_play_in
11,East,14,North Dakota St.,North Dakota St,False
28,South,7,Saint Mary's,Saint Mary's CA,False


- which rows are unmatched because they are play-ins
- which rows are unmatched because our name mapping is still wrong
Expected unmatched
These are okay for now because they are play-ins:

- Lehigh / PVAMU
- NC St. / Texas
- SMU / Miami OH

## Step 7: find the exact Kaggle team names for those two schools

In [554]:
m_teams[m_teams["TeamName"].str.contains("North Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
214,1315,North Dakota,2009,2026


In [555]:
m_teams[m_teams["TeamName"].str.contains("Mary", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
157,1258,Loy Marymount,1985,2026
167,1268,Maryland,1985,2026
190,1291,Mt St Mary's,1989,2026
287,1388,St Mary's CA,1985,2026
355,1456,William & Mary,1985,2026


In [556]:
m_teams[m_teams["TeamName"].str.contains("Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
194,1295,N Dakota St,2006,2026
214,1315,North Dakota,2009,2026
254,1355,S Dakota St,2006,2026
276,1377,South Dakota,2009,2026


## Step 8: I want to see whether Kaggle has:

N Dakota St
North Dakota
something else

In [557]:
m_teams[m_teams["TeamName"].str.contains("Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
194,1295,N Dakota St,2006,2026
214,1315,North Dakota,2009,2026
254,1355,S Dakota St,2006,2026
276,1377,South Dakota,2009,2026


In [558]:
name_map["Saint Mary's"] = "St Mary's CA"


## Step 9: update the mapping

In [559]:
name_map["North Dakota St."] = "N Dakota St"
name_map["Saint Mary's"] = "St Mary's CA"


## Step 10: rebuild the cleaned column and merge again

In [560]:
bracket_clean["team_clean"] = bracket_clean["team"].replace(name_map)

merged_clean = bracket_clean.merge(
    team_lookup,
    left_on="team_clean",
    right_on="TeamName",
    how="left"
)


## Step 11: check what is still unmatched

In [561]:
merged_clean[merged_clean["TeamID"].isna()][["region", "seed", "team", "team_clean", "is_play_in"]]


,region,seed,team,team_clean,is_play_in


## Step 12: split the bracket into two groups:
- normal bracket teams
- play-in rows

In [562]:
bracket_main = merged_clean[~merged_clean["is_play_in"]].copy()
bracket_play_in = merged_clean[merged_clean["is_play_in"]].copy()


In [563]:
bracket_main.shape, bracket_play_in.shape


((64, 7), (0, 7))

## YAY: I now have bracket_main: 61 normal bracket rows and bracket_play_in 3 play-in rows!!

## Step 13: create a 2026-only feature table

In [564]:
features_2026 = team_features_named[team_features_named["Season"] == 2026].copy()


### Why: my full feature table contains many years, my bracket is for 2026, and i want to get current-season features now

## Step 14: merge the bracket teams with 2026 features

In [565]:
bracket_with_features = bracket_main.merge(
    features_2026,
    on="TeamID",
    how="left",
    suffixes=("", "_feature")
)


## Step 15: inspect whether anything is missing

In [566]:
bracket_with_features[
    ["region", "seed", "team", "TeamID", "wins", "losses", "win_pct", "avg_scoring_margin"]
].head(20)


,region,seed,team,TeamID,wins,losses,win_pct,avg_scoring_margin
0,East,1,Duke,1181,27.0,2.0,0.931034,20.344828
1,East,16,Siena,1373,20.0,11.0,0.645161,4.354839
2,East,8,Ole Miss,1279,12.0,17.0,0.413793,-0.482759
3,East,9,TCU,1395,19.0,10.0,0.655172,6.413793
4,East,5,St. John's,1385,23.0,6.0,0.793103,11.758621
5,East,12,Northern Iowa,1320,18.0,12.0,0.600000,6.266667
6,East,4,Kansas,1242,21.0,8.0,0.724138,7.379310
7,East,13,Cal Baptist,1465,21.0,8.0,0.724138,4.068966
8,East,6,Louisville,1257,20.0,9.0,0.689655,13.724138
9,East,11,South Florida,1378,20.0,8.0,0.714286,9.821429


### Checking for missing feature values

In [567]:
bracket_with_features[
    bracket_with_features["wins"].isna()
][["region", "seed", "team", "TeamID"]]


,region,seed,team,TeamID


## An empty result means:

**every non-play-in 2026 bracket team successfully matched to your 2026 feature table**

That is a big checkpoint.

I now have:

- a clean bracket table
- working team ID mapping
- play-ins isolated
- 2026 season features attached to all main bracket teams
- no missing feature rows for the main field

## Step 16: inspect the 2026 bracket table with features

In [568]:
bracket_with_features[
    ["region", "seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]
].sort_values(["region", "seed"]).head(20)


,region,seed,team,wins,losses,win_pct,avg_scoring_margin
0,East,1,Duke,27.0,2.0,0.931034,20.344828
14,East,2,UConn,27.0,3.0,0.900000,13.533333
10,East,3,Michigan St.,24.0,5.0,0.827586,11.965517
6,East,4,Kansas,21.0,8.0,0.724138,7.379310
4,East,5,St. John's,23.0,6.0,0.793103,11.758621
8,East,6,Louisville,20.0,9.0,0.689655,13.724138
12,East,7,UCLA,19.0,10.0,0.655172,6.137931
2,East,8,Ole Miss,12.0,17.0,0.413793,-0.482759
3,East,9,TCU,19.0,10.0,0.655172,6.413793
13,East,10,UCF,20.0,8.0,0.714286,4.392857


## Step 17: build a first-round game list for one region, starting with the East

## Step 18: create a list of first-round seed pairings

In [569]:
first_round_pairs = [
    (1, 16),
    (8, 9),
    (5, 12),
    (4, 13),
    (6, 11),
    (3, 14),
    (7, 10),
    (2, 15),
]


## Step 19: filter the East region

In [570]:
east_2026 = bracket_with_features[bracket_with_features["region"] == "East"].copy()
east_2026 = east_2026.sort_values("seed")
east_2026[["seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]]


,seed,team,wins,losses,win_pct,avg_scoring_margin
0,1,Duke,27.0,2.0,0.931034,20.344828
14,2,UConn,27.0,3.0,0.900000,13.533333
10,3,Michigan St.,24.0,5.0,0.827586,11.965517
6,4,Kansas,21.0,8.0,0.724138,7.379310
4,5,St. John's,23.0,6.0,0.793103,11.758621
8,6,Louisville,20.0,9.0,0.689655,13.724138
12,7,UCLA,19.0,10.0,0.655172,6.137931
2,8,Ole Miss,12.0,17.0,0.413793,-0.482759
3,9,TCU,19.0,10.0,0.655172,6.413793
13,10,UCF,20.0,8.0,0.714286,4.392857


## Step 20: pull one matchup manually

In [571]:
team_1 = east_2026[east_2026["seed"] == 1]
team_16 = east_2026[east_2026["seed"] == 16]

pd.concat([team_1, team_16])[["seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]]


,seed,team,wins,losses,win_pct,avg_scoring_margin
0,1,Duke,27.0,2.0,0.931034,20.344828
1,16,Siena,20.0,11.0,0.645161,4.354839


## Step 21: build the matchup row for 1 vs 16

In [572]:
team_1_row = team_1.iloc[0]
team_16_row = team_16.iloc[0]

matchup_example = pd.DataFrame([{
    "region": "East",
    "seed_a": team_1_row["seed"],
    "team_a": team_1_row["team"],
    "seed_b": team_16_row["seed"],
    "team_b": team_16_row["team"],
    "seed_diff": team_1_row["seed"] - team_16_row["seed"],
    "win_pct_diff": team_1_row["win_pct"] - team_16_row["win_pct"],
    "scoring_margin_diff": team_1_row["avg_scoring_margin"] - team_16_row["avg_scoring_margin"],
}])

matchup_example


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,15.989989


## Doing one more manual matchup

In [573]:
team_8 = east_2026[east_2026["seed"] == 8]
team_9 = east_2026[east_2026["seed"] == 9]

team_8_row = team_8.iloc[0]
team_9_row = team_9.iloc[0]

matchup_8_9 = pd.DataFrame([{
    "region": "East",
    "seed_a": team_8_row["seed"],
    "team_a": team_8_row["team"],
    "seed_b": team_9_row["seed"],
    "team_b": team_9_row["team"],
    "seed_diff": team_8_row["seed"] - team_9_row["seed"],
    "win_pct_diff": team_8_row["win_pct"] - team_9_row["win_pct"],
    "scoring_margin_diff": team_8_row["avg_scoring_margin"] - team_9_row["avg_scoring_margin"],
}])

matchup_8_9


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,8,Ole Miss,9,TCU,-1,-0.241379,-6.896552


## Step 1: define the function shell

In [574]:
def build_first_round_matchups(region_df, region_name):
    rows = []

    for seed_a, seed_b in first_round_pairs:
        team_a_df = region_df[region_df["seed"] == seed_a]
        team_b_df = region_df[region_df["seed"] == seed_b]

        if team_a_df.empty or team_b_df.empty:
            continue

        team_a = team_a_df.iloc[0]
        team_b = team_b_df.iloc[0]

        rows.append({
            "region": region_name,
            "seed_a": team_a["seed"],
            "team_a": team_a["team"],
            "seed_b": team_b["seed"],
            "team_b": team_b["team"],
            "seed_diff": team_a["seed"] - team_b["seed"],
            "win_pct_diff": team_a["win_pct"] - team_b["win_pct"],
            "points_for_diff": team_a["avg_points_for"] - team_b["avg_points_for"],
            "points_against_diff": team_a["avg_points_against"] - team_b["avg_points_against"],
            "scoring_margin_diff": team_a["avg_scoring_margin"] - team_b["avg_scoring_margin"],
        })

    return pd.DataFrame(rows)




## What this means:

- region_df is something like east_2026
- region_name is "East"
- rows will collect one dictionary per game
- the loop goes through each seed pairing

# Step 2: test it on East

In [575]:
east_matchups = build_first_round_matchups(east_2026, "East")
east_matchups


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,12.021135,-3.968854,15.989989
1,East,8,Ole Miss,9,TCU,-1,-0.241379,-2.862069,4.034483,-6.896552
2,East,5,St. John's,12,Northern Iowa,-7,0.193103,14.251724,8.759770,5.491954
3,East,4,Kansas,13,Cal Baptist,-9,0.000000,3.965517,0.655172,3.310345
4,East,6,Louisville,11,South Florida,-5,-0.024631,-1.498768,-5.401478,3.902709
5,East,3,Michigan St.,14,North Dakota St.,-11,0.077586,-1.368227,-5.690887,4.322660
6,East,7,UCLA,10,UCF,-3,-0.059113,-4.349754,-6.094828,1.745074
7,East,2,UConn,15,Furman,-13,0.328571,4.271429,-6.976190,11.247619


## A few rows stand out:

Duke vs Siena
Very strong signal for Duke.

Ole Miss vs TCU
Negative win_pct_diff and negative scoring_margin_diff, which suggests TCU may actually be stronger despite being the 9 seed.

Louisville vs South Florida
Close matchup. The differences are small.

UCLA vs UCF
Another relatively close matchup.

In [576]:
south_2026 = bracket_with_features[bracket_with_features["region"] == "South"].copy()
west_2026 = bracket_with_features[bracket_with_features["region"] == "West"].copy()
midwest_2026 = bracket_with_features[bracket_with_features["region"] == "Midwest"].copy()


In [577]:
south_2026[["seed", "team"]].sort_values("seed")


,seed,team
16,1,Florida
30,2,Houston
26,3,Illinois
22,4,Nebraska
20,5,Vanderbilt
24,6,North Carolina
28,7,Saint Mary's
18,8,Clemson
19,9,Iowa
29,10,Texas A&M


## Step 3: rerun the region builds

In [578]:
south_matchups = build_first_round_matchups(south_2026, "South")
west_matchups = build_first_round_matchups(west_2026, "West")
midwest_matchups = build_first_round_matchups(midwest_2026, "Midwest")


In [579]:
first_round_2026 = pd.concat(
    [east_matchups, south_matchups, west_matchups, midwest_matchups],
    ignore_index=True
)

first_round_2026


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,12.021135,-3.968854,15.989989
1,East,8,Ole Miss,9,TCU,-1,-0.241379,-2.862069,4.034483,-6.896552
2,East,5,St. John's,12,Northern Iowa,-7,0.193103,14.251724,8.759770,5.491954
3,East,4,Kansas,13,Cal Baptist,-9,0.000000,3.965517,0.655172,3.310345
4,East,6,Louisville,11,South Florida,-5,-0.024631,-1.498768,-5.401478,3.902709
5,East,3,Michigan St.,14,North Dakota St.,-11,0.077586,-1.368227,-5.690887,4.322660
6,East,7,UCLA,10,UCF,-3,-0.059113,-4.349754,-6.094828,1.745074
7,East,2,UConn,15,Furman,-13,0.328571,4.271429,-6.976190,11.247619
8,South,1,Florida,16,PVAMU,-15,0.446950,10.368700,-10.586207,20.954907
9,South,8,Clemson,9,Iowa,-1,0.034483,-0.965517,0.896552,-1.862069


# Building historical tournament matchup rows for training

## Step 1: start with a tiny sample

In [580]:
tourney_sample = data["tourney"].head(5).copy()
tourney_sample


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


### What I'm looking at:

- each row is one real NCAA tournament game
- WTeamID is the winner
- LTeamID is the loser
- Season tells us which year’s team features to use

## Step 2: build the smaller feature table

In [581]:
team_feature_cols = [
    "Season",
    "TeamID",
    "win_pct",
    "avg_points_for",
    "avg_points_against",
    "avg_scoring_margin",
]

team_features_small = team_features_named[team_feature_cols].copy()
team_features_small.head()


,Season,TeamID,win_pct,avg_points_for,avg_points_against,avg_scoring_margin
0,1985,1102,0.208333,63.083333,68.875000,-5.791667
1,1985,1103,0.391304,61.043478,64.086957,-3.043478
2,1985,1104,0.700000,68.500000,60.700000,7.800000
3,1985,1106,0.416667,71.625000,75.416667,-3.791667
4,1985,1108,0.760000,83.000000,75.040000,7.960000


## Step 3: attach winner features

In [582]:
winner_features = team_features_small.add_prefix("team_a_")
winner_features = winner_features.rename(
    columns={
        "team_a_Season": "Season",
        "team_a_TeamID": "WTeamID",
    }
)

sample_with_winner = tourney_sample.merge(
    winner_features,
    on=["Season", "WTeamID"],
    how="left"
)

sample_with_winner.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,team_a_win_pct,team_a_avg_points_for,team_a_avg_points_against,team_a_avg_scoring_margin
0,1985,136,1116,63,1234,54,N,0,0.636364,65.333333,61.696970,3.636364
1,1985,136,1120,59,1345,58,N,0,0.620690,70.344828,66.655172,3.689655
2,1985,136,1207,68,1250,43,N,0,0.925926,75.740741,60.074074,15.666667
3,1985,136,1229,58,1425,55,N,0,0.740741,71.592593,65.629630,5.962963
4,1985,136,1242,49,1325,38,N,0,0.766667,76.033333,70.400000,5.633333


## Step 4: Prepare loser Features

In [583]:
loser_features = team_features_small.add_prefix("team_b_")
loser_features = loser_features.rename(
    columns={
        "team_b_Season": "Season",
        "team_b_TeamID": "LTeamID",
    }
)


## Step 5: merge loser features into the sample

In [584]:
sample_with_both = sample_with_winner.merge(
    loser_features,
    on=["Season", "LTeamID"],
    how="left"
)

sample_with_both.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,team_a_win_pct,team_a_avg_points_for,team_a_avg_points_against,team_a_avg_scoring_margin,team_b_win_pct,team_b_avg_points_for,team_b_avg_points_against,team_b_avg_scoring_margin
0,1985,136,1116,63,1234,54,N,0,0.636364,65.333333,61.696970,3.636364,0.666667,69.733333,59.266667,10.466667
1,1985,136,1120,59,1345,58,N,0,0.620690,70.344828,66.655172,3.689655,0.680000,69.120000,65.320000,3.800000
2,1985,136,1207,68,1250,43,N,0,0.925926,75.740741,60.074074,15.666667,0.379310,65.758621,70.206897,-4.448276
3,1985,136,1229,58,1425,55,N,0,0.740741,71.592593,65.629630,5.962963,0.678571,68.392857,64.607143,3.785714
4,1985,136,1242,49,1325,38,N,0,0.766667,76.033333,70.400000,5.633333,0.740741,67.555556,63.000000,4.555556


### I now see columns like:

- team_b_win_pct
- team_b_avg_points_for
- team_b_avg_points_against
- team_b_avg_scoring_margin

## What this means: Each row is one historic NCAA tournment game, and it includes winner and loser features

## Step 6: create diff columns

In [585]:
sample_with_both["win_pct_diff"] = (
    sample_with_both["team_a_win_pct"] - sample_with_both["team_b_win_pct"]
)

sample_with_both["points_for_diff"] = (
    sample_with_both["team_a_avg_points_for"] - sample_with_both["team_b_avg_points_for"]
)

sample_with_both["points_against_diff"] = (
    sample_with_both["team_a_avg_points_against"] - sample_with_both["team_b_avg_points_against"]
)

sample_with_both["scoring_margin_diff"] = (
    sample_with_both["team_a_avg_scoring_margin"] - sample_with_both["team_b_avg_scoring_margin"]
)


### Then inspect just the useful columns

In [586]:
sample_with_both[
    [
        "Season",
        "WTeamID",
        "LTeamID",
        "team_a_win_pct",
        "team_b_win_pct",
        "win_pct_diff",
        "team_a_avg_scoring_margin",
        "team_b_avg_scoring_margin",
        "scoring_margin_diff",
    ]
].head()


,Season,WTeamID,LTeamID,team_a_win_pct,team_b_win_pct,win_pct_diff,team_a_avg_scoring_margin,team_b_avg_scoring_margin,scoring_margin_diff
0,1985,1116,1234,0.636364,0.666667,-0.030303,3.636364,10.466667,-6.830303
1,1985,1120,1345,0.620690,0.680000,-0.059310,3.689655,3.800000,-0.110345
2,1985,1207,1250,0.925926,0.379310,0.546616,15.666667,-4.448276,20.114943
3,1985,1229,1425,0.740741,0.678571,0.062169,5.962963,3.785714,2.177249
4,1985,1242,1325,0.766667,0.740741,0.025926,5.633333,4.555556,1.077778


### Right now team_a is always the winner.
So if you trained a model on this table as-is, I’d be leaking the answer into the row construction.

What comes next is fixing that setup.

The clean idea is:

- create rows where team_a and team_b are assigned in a neutral way
- add a label like team_a_won = 1 or 0

## Take one historical game row

In [587]:
game = tourney_sample.iloc[0]
game


Season     1985
DayNum      136
WTeamID    1116
WScore       63
LTeamID    1234
LScore       54
WLoc          N
NumOT         0
Name: 0, dtype: object

### Decide who is side A and side B neutrally

In [588]:
team_a_id = min(game["WTeamID"], game["LTeamID"])
team_b_id = max(game["WTeamID"], game["LTeamID"])

team_a_won = int(team_a_id == game["WTeamID"])

team_a_id, team_b_id, team_a_won


(np.int64(1116), np.int64(1234), 1)

### Get side A and side B features from the season table

In [589]:
season = game["Season"]

team_a_features = team_features_small[
    (team_features_small["Season"] == season) &
    (team_features_small["TeamID"] == team_a_id)
].iloc[0]

team_b_features = team_features_small[
    (team_features_small["Season"] == season) &
    (team_features_small["TeamID"] == team_b_id)
].iloc[0]


### Build one neutral matchup row

In [590]:
neutral_matchup_example = pd.DataFrame([{
    "Season": season,
    "team_a_id": team_a_id,
    "team_b_id": team_b_id,
    "team_a_won": team_a_won,
    "seed_diff": 0,  # placeholder for now
    "win_pct_diff": team_a_features["win_pct"] - team_b_features["win_pct"],
    "points_for_diff": team_a_features["avg_points_for"] - team_b_features["avg_points_for"],
    "points_against_diff": team_a_features["avg_points_against"] - team_b_features["avg_points_against"],
    "scoring_margin_diff": team_a_features["avg_scoring_margin"] - team_b_features["avg_scoring_margin"],
}])

neutral_matchup_example


,Season,team_a_id,team_b_id,team_a_won,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,0,-0.030303,-4.4,2.430303,-6.830303


### What this row says
- team_a_id = 1116
- team_b_id = 1234
- team_a_won = 1
So side A won the game.

But the feature differences are mostly negative:

- win_pct_diff is negative
- points_for_diff is negative
- scoring_margin_diff is negative

## Take historical tournament games and turn them into a training table of neutral matchup rows

In [591]:
def build_historical_matchups(tourney_df, team_features_df):
    rows = []

    for _, game in tourney_df.iterrows():
        season = game["Season"]

        team_a_id = min(game["WTeamID"], game["LTeamID"])
        team_b_id = max(game["WTeamID"], game["LTeamID"])

        team_a_won = int(team_a_id == game["WTeamID"])

        team_a_features = team_features_df[
            (team_features_df["Season"] == season) &
            (team_features_df["TeamID"] == team_a_id)
        ]

        team_b_features = team_features_df[
            (team_features_df["Season"] == season) &
            (team_features_df["TeamID"] == team_b_id)
        ]

        if team_a_features.empty or team_b_features.empty:
            continue

        team_a_features = team_a_features.iloc[0]
        team_b_features = team_b_features.iloc[0]

        rows.append({
            "Season": season,
            "team_a_id": team_a_id,
            "team_b_id": team_b_id,
            "team_a_won": team_a_won,
            "win_pct_diff": team_a_features["win_pct"] - team_b_features["win_pct"],
            "points_for_diff": team_a_features["avg_points_for"] - team_b_features["avg_points_for"],
            "points_against_diff": team_a_features["avg_points_against"] - team_b_features["avg_points_against"],
            "scoring_margin_diff": team_a_features["avg_scoring_margin"] - team_b_features["avg_scoring_margin"],
        })

    return pd.DataFrame(rows)



### Test in on a tiny sample first

In [592]:
historical_sample_matchups = build_historical_matchups(tourney_sample, team_features_named)
historical_sample_matchups


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,-0.030303,-4.400000,2.430303,-6.830303
1,1985,1120,1345,1,-0.059310,1.224828,1.335172,-0.110345
2,1985,1207,1250,1,0.546616,9.982120,-10.132822,20.114943
3,1985,1229,1425,1,0.062169,3.199735,1.022487,2.177249
4,1985,1242,1325,1,0.025926,8.477778,7.400000,1.077778


### Build the full historical matchup table

In [593]:
historical_matchups = build_historical_matchups(data["tourney"], team_features_named)
historical_matchups.head()


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,-0.030303,-4.400000,2.430303,-6.830303
1,1985,1120,1345,1,-0.059310,1.224828,1.335172,-0.110345
2,1985,1207,1250,1,0.546616,9.982120,-10.132822,20.114943
3,1985,1229,1425,1,0.062169,3.199735,1.022487,2.177249
4,1985,1242,1325,1,0.025926,8.477778,7.400000,1.077778


In [594]:
historical_matchups.shape

(2585, 8)

In [595]:
historical_matchups["team_a_won"].value_counts()


team_a_won
1    1323
0    1262
Name: count, dtype: int64

### Confirmed:

how many historical tournament games turned into training rows
whether team_a_won has both classes:
- some 1
- some 0

In [596]:
historical_matchups.describe()


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
count,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000
mean,2004.908704,1227.195745,1347.791876,0.511799,0.002907,0.908370,0.352231,0.556139
std,11.764137,82.038896,83.703438,0.499957,0.150662,8.442246,6.939126,6.872930
min,1985.000000,1101.000000,1112.000000,0.000000,-0.633333,-52.946429,-47.379464,-27.125641
25%,1995.000000,1163.000000,1277.000000,0.000000,-0.093750,-4.227586,-3.956250,-3.906250
50%,2005.000000,1227.000000,1356.000000,1.000000,0.005974,0.981061,0.267992,0.250000
75%,2015.000000,1277.000000,1423.000000,1.000000,0.101326,6.346154,4.740530,5.022059
max,2025.000000,1458.000000,1471.000000,1.000000,0.611607,45.285714,37.428571,29.057576


### What I take from it:

- all feature columns are numeric
- no missing values in the training table
- both positive and negative differences exist, which is what you want
- team_a_won is close to balanced, which is also good

## Prepare Baseline Model Inputs

This section defines the first feature set for the baseline logistic regression model.

For the first pass, I am using:
- win percentage difference
- points scored difference
- points allowed difference
- scoring margin difference

These feature differences represent how much stronger or weaker team A looks compared to team B before the game.


In [597]:
feature_cols = [
    "win_pct_diff",
    "points_for_diff",
    "points_against_diff",
    "scoring_margin_diff",
]

X = historical_matchups[feature_cols]
y = historical_matchups["team_a_won"]


## Split Historical Matchup Data

This step splits the historical matchup dataset into training and test sets.

The model will learn from the training set and then be evaluated on the held-out test set. I used `stratify=y` so the balance of wins and losses for team A stays similar in both sets.


In [598]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## Import Logistic Regression and Evaluation Metrics

This section imports the baseline classification model and the metrics used to judge how well it performs on historical tournament matchups.


In [599]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss


## Train Baseline Logistic Regression Model

This step fits a logistic regression model on the historical matchup feature differences.

The goal is to predict whether team A wins the game based on the pregame feature differences between team A and team B.


In [600]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

## Generate Predictions on the Test Set

After training the model, this step creates:
- predicted classes (`0` or `1`)
- predicted probabilities that team A wins

These outputs are used to evaluate how useful the model is before applying it to the 2026 bracket.


In [601]:
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]


## Evaluate Baseline Model Performance

This section measures how well the baseline model performs on held-out historical tournament games.

I am looking at:
- accuracy
- log loss
- classification report
- confusion matrix


In [602]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log loss:", log_loss(y_test, y_prob))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.690522243713733
Log loss: 0.5823483028815494

Classification report:
               precision    recall  f1-score   support

           0       0.68      0.68      0.68       252
           1       0.70      0.70      0.70       265

    accuracy                           0.69       517
   macro avg       0.69      0.69      0.69       517
weighted avg       0.69      0.69      0.69       517


Confusion matrix:
 [[171  81]
 [ 79 186]]


## What These Results Mean

The baseline logistic regression reached about **69% accuracy**, which means it predicted the winner correctly in about 69 out of every 100 held-out historical tournament games.

The **log loss** of about **0.58** suggests the predicted probabilities are reasonably informative, not just the hard win/loss guesses. Lower log loss is better, so this gives me a starting point for judging whether future feature engineering improves the model.

The classification report shows that the model is performing fairly evenly across both classes (`team_a_won = 0` and `team_a_won = 1`), which is a good sign that it is not heavily biased toward one side.

The confusion matrix shows:
- `171` games where team A was predicted to lose and actually lost
- `186` games where team A was predicted to win and actually won
- `81` cases where the model predicted team A would win but team A lost
- `79` cases where the model predicted team A would lose but team A won

Overall, this is a solid baseline. It shows that the current feature set contains real predictive signal, even before adding more advanced features or full bracket simulation.


## Generate win probabilities for the current bracket matchups


## Use the same feature columns on 2026 matchups

In [603]:
X_2026 = first_round_2026[feature_cols]
X_2026.head()


,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,0.285873,12.021135,-3.968854,15.989989
1,-0.241379,-2.862069,4.034483,-6.896552
2,0.193103,14.251724,8.759770,5.491954
3,0.000000,3.965517,0.655172,3.310345
4,-0.024631,-1.498768,-5.401478,3.902709


## Score the 2026 first-round games

In [604]:
first_round_2026["team_a_win_prob"] = log_reg.predict_proba(X_2026)[:, 1]
first_round_2026["team_b_win_prob"] = 1 - first_round_2026["team_a_win_prob"]


## Convert Probabilities Into Predicted Winners

This step turns the model probabilities into a bracket-style prediction by choosing the team with the higher win probability in each matchup.


## View First-Round Prediction Output

This table shows the baseline model's round 1 predictions for the current bracket, including both team win probabilities and the predicted winner.


In [605]:
first_round_2026["predicted_winner"] = first_round_2026.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)


In [606]:
first_round_2026[
    ["region", "team_a", "team_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,team_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,East,Duke,Siena,0.896083,0.103917,Duke
1,East,Ole Miss,TCU,0.282931,0.717069,TCU
2,East,St. John's,Northern Iowa,0.687494,0.312506,St. John's
3,East,Kansas,Cal Baptist,0.609764,0.390236,Kansas
4,East,Louisville,South Florida,0.619817,0.380183,Louisville
5,East,Michigan St.,North Dakota St.,0.630652,0.369348,Michigan St.
6,East,UCLA,UCF,0.546426,0.453574,UCLA
7,East,UConn,Furman,0.812610,0.187390,UConn
8,South,Florida,PVAMU,0.941828,0.058172,Florida
9,South,Clemson,Iowa,0.431105,0.568895,Iowa


### A few interesting things jump out:

- Duke, Arizona, Illinois, Houston, Michigan, Iowa St. look like strong favorites
- model likes TCU over Ole Miss
- it likes VCU over North Carolina
- it likes High Point over Wisconsin
- it likes Akron over Texas Tech
- it likes Santa Clara over Kentucky


## Identify the closest game and strongest upset picks

In [607]:
first_round_2026["favorite_win_prob"] = first_round_2026[
    ["team_a_win_prob", "team_b_win_prob"]
].max(axis=1)

first_round_2026["underdog_pick"] = first_round_2026["predicted_winner"] != first_round_2026["team_a"]

first_round_2026.sort_values("favorite_win_prob").head(10)


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff,team_a_win_prob,team_b_win_prob,predicted_winner,favorite_win_prob,underdog_pick
12,South,6,North Carolina,11,VCU,-5,0.034483,-2.448276,-1.655172,-0.793103,0.463528,0.536472,VCU,0.536472,True
6,East,7,UCLA,10,UCF,-3,-0.059113,-4.349754,-6.094828,1.745074,0.546426,0.453574,UCLA,0.546426,False
28,Midwest,6,Tennessee,11,Miami OH,-5,-0.310345,-6.931034,-5.677719,-1.253316,0.449202,0.550798,Miami OH,0.550798,True
20,West,6,BYU,11,Texas,-5,0.082512,1.894089,-0.116995,2.011084,0.562474,0.437526,BYU,0.562474,False
19,West,4,Arkansas,13,Hawaii,-9,-0.006631,11.139257,9.501326,1.637931,0.568555,0.431445,Arkansas,0.568555,False
9,South,8,Clemson,9,Iowa,-1,0.034483,-0.965517,0.896552,-1.862069,0.431105,0.568895,Iowa,0.568895,True
30,Midwest,7,Kentucky,10,Santa Clara,-3,-0.111494,-2.558621,-0.480460,-2.078161,0.424956,0.575044,Santa Clara,0.575044,True
14,South,7,Saint Mary's,10,Texas A&M,-3,0.211494,-10.235632,-13.928736,3.693103,0.592795,0.407205,Saint Mary's,0.592795,False
26,Midwest,5,Texas Tech,12,Akron,-7,-0.056194,-6.187739,-3.756066,-2.431673,0.406010,0.593990,Akron,0.593990,True
10,South,5,Vanderbilt,12,McNeese,-7,-0.062808,10.217980,7.588670,2.629310,0.600251,0.399749,Vanderbilt,0.600251,False


In [608]:
first_round_2026[first_round_2026["underdog_pick"]]


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff,team_a_win_prob,team_b_win_prob,predicted_winner,favorite_win_prob,underdog_pick
1,East,8,Ole Miss,9,TCU,-1,-0.241379,-2.862069,4.034483,-6.896552,0.282931,0.717069,TCU,0.717069,True
9,South,8,Clemson,9,Iowa,-1,0.034483,-0.965517,0.896552,-1.862069,0.431105,0.568895,Iowa,0.568895,True
12,South,6,North Carolina,11,VCU,-5,0.034483,-2.448276,-1.655172,-0.793103,0.463528,0.536472,VCU,0.536472,True
17,West,8,Villanova,9,Utah St.,-1,-0.062808,-5.149015,0.472906,-5.621921,0.311045,0.688955,Utah St.,0.688955,True
18,West,5,Wisconsin,12,High Point,-7,-0.167488,-3.927340,3.891626,-7.818966,0.255940,0.744060,High Point,0.744060,True
25,Midwest,8,Georgia,9,Saint Louis,-1,-0.203202,1.438424,8.415025,-6.976601,0.286159,0.713841,Saint Louis,0.713841,True
26,Midwest,5,Texas Tech,12,Akron,-7,-0.056194,-6.187739,-3.756066,-2.431673,0.406010,0.593990,Akron,0.593990,True
28,Midwest,6,Tennessee,11,Miami OH,-5,-0.310345,-6.931034,-5.677719,-1.253316,0.449202,0.550798,Miami OH,0.550798,True
30,Midwest,7,Kentucky,10,Santa Clara,-3,-0.111494,-2.558621,-0.480460,-2.078161,0.424956,0.575044,Santa Clara,0.575044,True


## Closest games
The first table, sorted by favorite_win_prob, shows the least certain matchups. Those are the games where the model thinks either team could realistically win.

Underdog picks
The second table shows where the model is picking team_b over team_a, which in your current setup means it is picking the nominal lower side in the row, often the worse seed.

A few clear takeaways:

- VCU over North Carolina is a very close upset signal
- High Point over Wisconsin is a stronger upset signal
- Akron over Texas Tech is another upset signal
- Santa Clara over Kentucky is a moderate upset signal
- TCU over Ole Miss, Iowa over Clemson, Utah St. over Villanova, Saint Louis over Georgia are basically coin-flip style matchups where the model leans to the other side

In [609]:
first_round_2026["predicted_seed"] = first_round_2026.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

first_round_2026["expected_seed"] = first_round_2026[["seed_a", "seed_b"]].min(axis=1)

first_round_2026["is_upset_pick"] = first_round_2026["predicted_seed"] > first_round_2026["expected_seed"]


In [610]:
first_round_2026[first_round_2026["is_upset_pick"]][
    ["region", "team_a", "seed_a", "team_b", "seed_b", "predicted_winner", "team_a_win_prob", "team_b_win_prob"]
]


,region,team_a,seed_a,team_b,seed_b,predicted_winner,team_a_win_prob,team_b_win_prob
1,East,Ole Miss,8,TCU,9,TCU,0.282931,0.717069
9,South,Clemson,8,Iowa,9,Iowa,0.431105,0.568895
12,South,North Carolina,6,VCU,11,VCU,0.463528,0.536472
17,West,Villanova,8,Utah St.,9,Utah St.,0.311045,0.688955
18,West,Wisconsin,5,High Point,12,High Point,0.255940,0.744060
25,Midwest,Georgia,8,Saint Louis,9,Saint Louis,0.286159,0.713841
26,Midwest,Texas Tech,5,Akron,12,Akron,0.406010,0.593990
28,Midwest,Tennessee,6,Miami OH,11,Miami OH,0.449202,0.550798
30,Midwest,Kentucky,7,Santa Clara,10,Santa Clara,0.424956,0.575044


### Prediction Table
## Build a Clean Round 1 Prediction Table

This section creates a cleaner table for the 2026 first-round predictions.

The goal is to make the model output easier to read by keeping only the most useful columns:
- teams
- seeds
- win probabilities
- predicted winner
- favorite


In [611]:
round1_predictions = first_round_2026[
    [
        "region",
        "team_a",
        "seed_a",
        "team_b",
        "seed_b",
        "team_a_win_prob",
        "team_b_win_prob",
        "predicted_winner",
    ]
].copy()

round1_predictions


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,East,Duke,1,Siena,16,0.896083,0.103917,Duke
1,East,Ole Miss,8,TCU,9,0.282931,0.717069,TCU
2,East,St. John's,5,Northern Iowa,12,0.687494,0.312506,St. John's
3,East,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas
4,East,Louisville,6,South Florida,11,0.619817,0.380183,Louisville
5,East,Michigan St.,3,North Dakota St.,14,0.630652,0.369348,Michigan St.
6,East,UCLA,7,UCF,10,0.546426,0.453574,UCLA
7,East,UConn,2,Furman,15,0.812610,0.187390,UConn
8,South,Florida,1,PVAMU,16,0.941828,0.058172,Florida
9,South,Clemson,8,Iowa,9,0.431105,0.568895,Iowa


## Add a Favorite Column

This step adds a simple favorite label so the round 1 prediction table is easier to scan.


In [612]:
round1_predictions["favorite"] = round1_predictions.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= row["team_b_win_prob"] else row["team_b"],
    axis=1
)

round1_predictions


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite
0,East,Duke,1,Siena,16,0.896083,0.103917,Duke,Duke
1,East,Ole Miss,8,TCU,9,0.282931,0.717069,TCU,TCU
2,East,St. John's,5,Northern Iowa,12,0.687494,0.312506,St. John's,St. John's
3,East,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas,Kansas
4,East,Louisville,6,South Florida,11,0.619817,0.380183,Louisville,Louisville
5,East,Michigan St.,3,North Dakota St.,14,0.630652,0.369348,Michigan St.,Michigan St.
6,East,UCLA,7,UCF,10,0.546426,0.453574,UCLA,UCLA
7,East,UConn,2,Furman,15,0.812610,0.187390,UConn,UConn
8,South,Florida,1,PVAMU,16,0.941828,0.058172,Florida,Florida
9,South,Clemson,8,Iowa,9,0.431105,0.568895,Iowa,Iowa


## What the Round 1 Upset Picks Suggest

The model did surface several underdog-style picks in the first round. These are the games where the predicted winner was not the more expected team by the way the matchup was set up, which makes them useful early candidates for upset discussion.

Some of these look more like true coin-flip games, while others look like stronger upset signals. Based on the current baseline model, examples of underdog-style picks include:
- TCU over Ole Miss
- Iowa over Clemson
- VCU over North Carolina
- Utah St. over Villanova
- High Point over Wisconsin
- Saint Louis over Georgia
- Akron over Texas Tech
- Santa Clara over Kentucky

These picks should not be treated as final answers yet, but they do show that the current feature set and baseline model are already finding places where team quality may disagree with seed expectations.


## Fixed 2 of the play-in results in my data and just checking if its good. Still waiting for the last 2 games.

In [613]:
first_round_2026.shape


(32, 18)

In [614]:
west_matchups


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,West,1,Arizona,16,Long Island,-15,0.253615,12.972191,-2.895439,15.867631
1,West,8,Villanova,9,Utah St.,-1,-0.062808,-5.149015,0.472906,-5.621921
2,West,5,Wisconsin,12,High Point,-7,-0.167488,-3.927340,3.891626,-7.818966
3,West,4,Arkansas,13,Hawaii,-9,-0.006631,11.139257,9.501326,1.637931
4,West,6,BYU,11,Texas,-5,0.082512,1.894089,-0.116995,2.011084
5,West,3,Gonzaga,14,Kennesaw St.,-11,0.326303,5.044665,-12.741935,17.786600
6,West,7,Miami (FL),10,Missouri,-3,0.103448,2.379310,-4.620690,7.000000
7,West,2,Purdue,15,Queens (N.C.),-13,0.191954,-2.356322,-13.568966,11.212644


## Bracket Advancement Logic

This section starts the bracket progression part of the project.

The goal is to take predicted winners from one round and use them to build the next round's matchups. I am starting with one region first so the bracket flow is easier to understand before extending it to the full tournament.



### Pull East round-1 predictions

In [615]:
east_round1 = round1_predictions[round1_predictions["region"] == "East"].copy()
east_round1

,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite
0,East,Duke,1,Siena,16,0.896083,0.103917,Duke,Duke
1,East,Ole Miss,8,TCU,9,0.282931,0.717069,TCU,TCU
2,East,St. John's,5,Northern Iowa,12,0.687494,0.312506,St. John's,St. John's
3,East,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas,Kansas
4,East,Louisville,6,South Florida,11,0.619817,0.380183,Louisville,Louisville
5,East,Michigan St.,3,North Dakota St.,14,0.630652,0.369348,Michigan St.,Michigan St.
6,East,UCLA,7,UCF,10,0.546426,0.453574,UCLA,UCLA
7,East,UConn,2,Furman,15,0.812610,0.187390,UConn,UConn


### Define round-2 pairing logic

In [616]:
second_round_pairs = [
    (0, 1),
    (2, 3),
    (4, 5),
    (6, 7),
]


### Winner table for East round 1

In [617]:
east_round1_winners = east_round1[
    ["team_a", "seed_a", "team_b", "seed_b", "predicted_winner"]
].copy()

east_round1_winners["winner_seed"] = east_round1_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round1_winners


,team_a,seed_a,team_b,seed_b,predicted_winner,winner_seed
0,Duke,1,Siena,16,Duke,1
1,Ole Miss,8,TCU,9,TCU,9
2,St. John's,5,Northern Iowa,12,St. John's,5
3,Kansas,4,Cal Baptist,13,Kansas,4
4,Louisville,6,South Florida,11,Louisville,6
5,Michigan St.,3,North Dakota St.,14,Michigan St.,3
6,UCLA,7,UCF,10,UCLA,7
7,UConn,2,Furman,15,UConn,2


In [618]:
first_round_2026.columns.tolist()


['region',
 'seed_a',
 'team_a',
 'seed_b',
 'team_b',
 'seed_diff',
 'win_pct_diff',
 'points_for_diff',
 'points_against_diff',
 'scoring_margin_diff',
 'team_a_win_prob',
 'team_b_win_prob',
 'predicted_winner',
 'favorite_win_prob',
 'underdog_pick',
 'predicted_seed',
 'expected_seed',
 'is_upset_pick']

In [619]:
bracket_with_features.columns.tolist()


['region',
 'seed',
 'team',
 'team_clean',
 'is_play_in',
 'TeamID',
 'TeamName',
 'Season',
 'wins',
 'losses',
 'games',
 'win_pct',
 'avg_points_for',
 'avg_points_against',
 'avg_scoring_margin',
 'TeamName_feature',
 'Seed']

### small team lookup table

In [620]:
team_lookup = bracket_with_features[
    ["region", "seed", "team", "TeamID"]
].copy()

team_lookup.head()


,region,seed,team,TeamID
0,East,1,Duke,1181
1,East,16,Siena,1373
2,East,8,Ole Miss,1279
3,East,9,TCU,1395
4,East,5,St. John's,1385


### attach teamid for side a in east round 1

In [621]:
east_round1_ids = east_round1.merge(
    team_lookup.rename(columns={"team": "team_a", "seed": "seed_a", "TeamID": "team_a_id"}),
    on=["region", "team_a", "seed_a"],
    how="left"
)

east_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id
0,East,Duke,1,Siena,16,0.896083,0.103917,Duke,Duke,1181
1,East,Ole Miss,8,TCU,9,0.282931,0.717069,TCU,TCU,1279
2,East,St. John's,5,Northern Iowa,12,0.687494,0.312506,St. John's,St. John's,1385
3,East,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas,Kansas,1242
4,East,Louisville,6,South Florida,11,0.619817,0.380183,Louisville,Louisville,1257


### attach teamid for side b

In [622]:
east_round1_ids = east_round1_ids.merge(
    team_lookup.rename(columns={"team": "team_b", "seed": "seed_b", "TeamID": "team_b_id"}),
    on=["region", "team_b", "seed_b"],
    how="left"
)

east_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id,team_b_id
0,East,Duke,1,Siena,16,0.896083,0.103917,Duke,Duke,1181,1373
1,East,Ole Miss,8,TCU,9,0.282931,0.717069,TCU,TCU,1279,1395
2,East,St. John's,5,Northern Iowa,12,0.687494,0.312506,St. John's,St. John's,1385,1320
3,East,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas,Kansas,1242,1465
4,East,Louisville,6,South Florida,11,0.619817,0.380183,Louisville,Louisville,1257,1378


### Create winner name, winner seed, winner teamid

In [623]:
east_round1_winners = east_round1_ids[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

east_round1_winners["winner_seed"] = east_round1_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round1_winners["winner_team_id"] = east_round1_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round1_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,East,Duke,1,1181,Siena,16,1373,Duke,1,1181
1,East,Ole Miss,8,1279,TCU,9,1395,TCU,9,1395
2,East,St. John's,5,1385,Northern Iowa,12,1320,St. John's,5,1385
3,East,Kansas,4,1242,Cal Baptist,13,1465,Kansas,4,1242
4,East,Louisville,6,1257,South Florida,11,1378,Louisville,6,1257
5,East,Michigan St.,3,1277,North Dakota St.,14,1295,Michigan St.,3,1277
6,East,UCLA,7,1417,UCF,10,1416,UCLA,7,1417
7,East,UConn,2,1163,Furman,15,1202,UConn,2,1163


### Build East round 2 with winner teamID included

In [624]:
east_round2_rows = []

for idx_a, idx_b in second_round_pairs:
    game_a = east_round1_winners.iloc[idx_a]
    game_b = east_round1_winners.iloc[idx_b]

    east_round2_rows.append({
        "region": "East",
        "team_a": game_a["predicted_winner"],
        "seed_a": game_a["winner_seed"],
        "team_a_id": game_a["winner_team_id"],
        "team_b": game_b["predicted_winner"],
        "seed_b": game_b["winner_seed"],
        "team_b_id": game_b["winner_team_id"],
    })

east_round2 = pd.DataFrame(east_round2_rows)
east_round2


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,East,Duke,1,1181,TCU,9,1395
1,East,St. John's,5,1385,Kansas,4,1242
2,East,Louisville,6,1257,Michigan St.,3,1277
3,East,UCLA,7,1417,UConn,2,1163


### Merge round 2 features using teamID

In [625]:
east_round2 = east_round2.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)


### Merge team B features

In [626]:
east_round2 = east_round2.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)


### Check for missing values

In [627]:
east_round2[
    ["team_a", "team_a_id", "team_a_TeamName", "team_b", "team_b_id", "team_b_TeamName"]
]


,team_a,team_a_id,team_a_TeamName,team_b,team_b_id,team_b_TeamName
0,Duke,1181,Duke,TCU,1395,TCU
1,St. John's,1385,St John's,Kansas,1242,Kansas
2,Louisville,1257,Louisville,Michigan St.,1277,Michigan St
3,UCLA,1417,UCLA,UConn,1163,Connecticut


### Build the round 2 diff columns

In [628]:
east_round2["win_pct_diff"] = east_round2["team_a_win_pct"] - east_round2["team_b_win_pct"]
east_round2["points_for_diff"] = east_round2["team_a_avg_points_for"] - east_round2["team_b_avg_points_for"]
east_round2["points_against_diff"] = east_round2["team_a_avg_points_against"] - east_round2["team_b_avg_points_against"]
east_round2["scoring_margin_diff"] = east_round2["team_a_avg_scoring_margin"] - east_round2["team_b_avg_scoring_margin"]


### Score East round 2 with the model

In [629]:
X_east_round2 = east_round2[feature_cols]

east_round2["team_a_win_prob"] = log_reg.predict_proba(X_east_round2)[:, 1]
east_round2["team_b_win_prob"] = 1 - east_round2["team_a_win_prob"]

east_round2["predicted_winner"] = east_round2.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

east_round2[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,East,Duke,1,TCU,9,0.861756,0.138244,Duke
1,East,St. John's,5,Kansas,4,0.645315,0.354685,St. John's
2,East,Louisville,6,Michigan St.,3,0.569706,0.430294,Louisville
3,East,UCLA,7,UConn,2,0.272507,0.727493,UConn


### What the East results say
Model currently has:

- Duke over TCU
- St. John's over Kansas
- Louisville over Michigan St.
- UConn over UCLA
A couple of those are straightforward:

- Duke over TCU
- UConn over UCLA
A couple are more interesting:

- St. John's over Kansas
- Louisville over Michigan St.

### Make the East round 2 winners table

In [630]:
east_round2_winners = east_round2[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

east_round2_winners["winner_seed"] = east_round2_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round2_winners["winner_team_id"] = east_round2_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round2_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,East,Duke,1,1181,TCU,9,1395,Duke,1,1181
1,East,St. John's,5,1385,Kansas,4,1242,St. John's,5,1385
2,East,Louisville,6,1257,Michigan St.,3,1277,Louisville,6,1257
3,East,UCLA,7,1417,UConn,2,1163,UConn,2,1163


### Build East round 3 matchups

In [631]:
east_round3_rows = []

east_round3_rows.append({
    "region": "East",
    "team_a": east_round2_winners.iloc[0]["predicted_winner"],
    "seed_a": east_round2_winners.iloc[0]["winner_seed"],
    "team_a_id": east_round2_winners.iloc[0]["winner_team_id"],
    "team_b": east_round2_winners.iloc[1]["predicted_winner"],
    "seed_b": east_round2_winners.iloc[1]["winner_seed"],
    "team_b_id": east_round2_winners.iloc[1]["winner_team_id"],
})

east_round3_rows.append({
    "region": "East",
    "team_a": east_round2_winners.iloc[2]["predicted_winner"],
    "seed_a": east_round2_winners.iloc[2]["winner_seed"],
    "team_a_id": east_round2_winners.iloc[2]["winner_team_id"],
    "team_b": east_round2_winners.iloc[3]["predicted_winner"],
    "seed_b": east_round2_winners.iloc[3]["winner_seed"],
    "team_b_id": east_round2_winners.iloc[3]["winner_team_id"],
})

east_round3 = pd.DataFrame(east_round3_rows)
east_round3


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,East,Duke,1,1181,St. John's,5,1385
1,East,Louisville,6,1257,UConn,2,1163


### Merge team A features by TeamId

In [632]:
east_round3 = east_round3.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)


### Merge team B features by TeamId

In [633]:
east_round3 = east_round3.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)


### Make the diff columns

In [634]:
east_round3["win_pct_diff"] = east_round3["team_a_win_pct"] - east_round3["team_b_win_pct"]
east_round3["points_for_diff"] = east_round3["team_a_avg_points_for"] - east_round3["team_b_avg_points_for"]
east_round3["points_against_diff"] = east_round3["team_a_avg_points_against"] - east_round3["team_b_avg_points_against"]
east_round3["scoring_margin_diff"] = east_round3["team_a_avg_scoring_margin"] - east_round3["team_b_avg_scoring_margin"]


### Score East round 3

In [635]:
X_east_round3 = east_round3[feature_cols]

east_round3["team_a_win_prob"] = log_reg.predict_proba(X_east_round3)[:, 1]
east_round3["team_b_win_prob"] = 1 - east_round3["team_a_win_prob"]

east_round3["predicted_winner"] = east_round3.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

east_round3[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,East,Duke,1,St. John's,5,0.751286,0.248714,Duke
1,East,Louisville,6,UConn,2,0.519357,0.480643,Louisville


### This means the East regional final should be:

- Duke
- Louisville

### Build East regional final

In [636]:
east_round3_winners = east_round3[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

east_round3_winners["winner_seed"] = east_round3_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round3_winners["winner_team_id"] = east_round3_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round3_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,East,Duke,1,1181,St. John's,5,1385,Duke,1,1181
1,East,Louisville,6,1257,UConn,2,1163,Louisville,6,1257


In [637]:
east_final = pd.DataFrame([{
    "region": "East",
    "team_a": east_round3_winners.iloc[0]["predicted_winner"],
    "seed_a": east_round3_winners.iloc[0]["winner_seed"],
    "team_a_id": east_round3_winners.iloc[0]["winner_team_id"],
    "team_b": east_round3_winners.iloc[1]["predicted_winner"],
    "seed_b": east_round3_winners.iloc[1]["winner_seed"],
    "team_b_id": east_round3_winners.iloc[1]["winner_team_id"],
}])

east_final


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,East,Duke,1,1181,Louisville,6,1257


In [638]:
east_final.T


,0
region,East
team_a,Duke
seed_a,1
team_a_id,1181
team_b,Louisville
seed_b,6
team_b_id,1257


In [639]:
east_final = east_final.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

east_final = east_final.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)


In [640]:
east_final["win_pct_diff"] = east_final["team_a_win_pct"] - east_final["team_b_win_pct"]
east_final["points_for_diff"] = east_final["team_a_avg_points_for"] - east_final["team_b_avg_points_for"]
east_final["points_against_diff"] = east_final["team_a_avg_points_against"] - east_final["team_b_avg_points_against"]
east_final["scoring_margin_diff"] = east_final["team_a_avg_scoring_margin"] - east_final["team_b_avg_scoring_margin"]


In [641]:
X_east_final = east_final[feature_cols]

east_final["team_a_win_prob"] = log_reg.predict_proba(X_east_final)[:, 1]
east_final["team_b_win_prob"] = 1 - east_final["team_a_win_prob"]

east_final["predicted_winner"] = east_final.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

east_final[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,East,Duke,1,Louisville,6,0.692216,0.307784,Duke


## East Region Result

The model now gives a full predicted path through the East region. This is the first full example of round-by-round bracket advancement working correctly from the current baseline model.

### Predicted East path
- Round 1 winners: Duke, TCU, St. John's, Kansas, Louisville, Michigan St., UCLA, UConn
- Round 2 winners: Duke, St. John's, Louisville, UConn
- Round 3 winners: Duke, Louisville
- East champion: Duke

### Why this matters
This is an important milestone because it shows the full regional bracket flow works:
- winners can be advanced from one round to the next
- `TeamID` can be carried forward instead of relying only on names
- later-round matchup features can be rebuilt correctly
- the model can be used again at each round instead of stopping after round 1

This East example is the pattern that can now be repeated for the other regions and eventually for the Final Four and championship game.


### Pulling West Round-1 predictions

In [642]:
west_round1 = round1_predictions[round1_predictions["region"] == "West"].copy()
west_round1


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite
16,West,Arizona,1,Long Island,16,0.895500,0.104500,Arizona,Arizona
17,West,Villanova,8,Utah St.,9,0.311045,0.688955,Utah St.,Utah St.
18,West,Wisconsin,5,High Point,12,0.255940,0.744060,High Point,High Point
19,West,Arkansas,4,Hawaii,13,0.568555,0.431445,Arkansas,Arkansas
20,West,BYU,6,Texas,11,0.562474,0.437526,BYU,BYU
21,West,Gonzaga,3,Kennesaw St.,14,0.912075,0.087925,Gonzaga,Gonzaga
22,West,Miami (FL),7,Missouri,10,0.713737,0.286263,Miami (FL),Miami (FL)
23,West,Purdue,2,Queens (N.C.),15,0.806823,0.193177,Purdue,Purdue


### Attach TeamID for side A

In [643]:
west_round1_ids = west_round1.merge(
    team_lookup.rename(columns={"team": "team_a", "seed": "seed_a", "TeamID": "team_a_id"}),
    on=["region", "team_a", "seed_a"],
    how="left"
)

west_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id
0,West,Arizona,1,Long Island,16,0.895500,0.104500,Arizona,Arizona,1112
1,West,Villanova,8,Utah St.,9,0.311045,0.688955,Utah St.,Utah St.,1437
2,West,Wisconsin,5,High Point,12,0.255940,0.744060,High Point,High Point,1458
3,West,Arkansas,4,Hawaii,13,0.568555,0.431445,Arkansas,Arkansas,1116
4,West,BYU,6,Texas,11,0.562474,0.437526,BYU,BYU,1140


### Attach TeamId for side B

In [644]:
west_round1_ids = west_round1_ids.merge(
    team_lookup.rename(columns={"team": "team_b", "seed": "seed_b", "TeamID": "team_b_id"}),
    on=["region", "team_b", "seed_b"],
    how="left"
)

west_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id,team_b_id
0,West,Arizona,1,Long Island,16,0.895500,0.104500,Arizona,Arizona,1112,1254
1,West,Villanova,8,Utah St.,9,0.311045,0.688955,Utah St.,Utah St.,1437,1429
2,West,Wisconsin,5,High Point,12,0.255940,0.744060,High Point,High Point,1458,1219
3,West,Arkansas,4,Hawaii,13,0.568555,0.431445,Arkansas,Arkansas,1116,1218
4,West,BYU,6,Texas,11,0.562474,0.437526,BYU,BYU,1140,1400


### Build the Weat round-1 winners table

In [645]:
west_round1_winners = west_round1_ids[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

west_round1_winners["winner_seed"] = west_round1_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

west_round1_winners["winner_team_id"] = west_round1_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

west_round1_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,West,Arizona,1,1112,Long Island,16,1254,Arizona,1,1112
1,West,Villanova,8,1437,Utah St.,9,1429,Utah St.,9,1429
2,West,Wisconsin,5,1458,High Point,12,1219,High Point,12,1219
3,West,Arkansas,4,1116,Hawaii,13,1218,Arkansas,4,1116
4,West,BYU,6,1140,Texas,11,1400,BYU,6,1140
5,West,Gonzaga,3,1211,Kennesaw St.,14,1244,Gonzaga,3,1211
6,West,Miami (FL),7,1274,Missouri,10,1281,Miami (FL),7,1274
7,West,Purdue,2,1345,Queens (N.C.),15,1474,Purdue,2,1345


### Build West round 2

In [646]:
west_round2_rows = []

for idx_a, idx_b in second_round_pairs:
    game_a = west_round1_winners.iloc[idx_a]
    game_b = west_round1_winners.iloc[idx_b]

    west_round2_rows.append({
        "region": "West",
        "team_a": game_a["predicted_winner"],
        "seed_a": game_a["winner_seed"],
        "team_a_id": game_a["winner_team_id"],
        "team_b": game_b["predicted_winner"],
        "seed_b": game_b["winner_seed"],
        "team_b_id": game_b["winner_team_id"],
    })

west_round2 = pd.DataFrame(west_round2_rows)
west_round2


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,West,Arizona,1,1112,Utah St.,9,1429
1,West,High Point,12,1219,Arkansas,4,1116
2,West,BYU,6,1140,Gonzaga,3,1211
3,West,Miami (FL),7,1274,Purdue,2,1345


### Merge features by TeamID

In [647]:
west_round2 = west_round2.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

west_round2 = west_round2.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)


### Build diff columns

In [648]:
west_round2["win_pct_diff"] = west_round2["team_a_win_pct"] - west_round2["team_b_win_pct"]
west_round2["points_for_diff"] = west_round2["team_a_avg_points_for"] - west_round2["team_b_avg_points_for"]
west_round2["points_against_diff"] = west_round2["team_a_avg_points_against"] - west_round2["team_b_avg_points_against"]
west_round2["scoring_margin_diff"] = west_round2["team_a_avg_scoring_margin"] - west_round2["team_b_avg_scoring_margin"]


### Score West round 2

In [649]:
X_west_round2 = west_round2[feature_cols]

west_round2["team_a_win_prob"] = log_reg.predict_proba(X_west_round2)[:, 1]
west_round2["team_b_win_prob"] = 1 - west_round2["team_a_win_prob"]

west_round2["predicted_winner"] = west_round2.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

west_round2[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,West,Arizona,1,Utah St.,9,0.700953,0.299047,Arizona
1,West,High Point,12,Arkansas,4,0.635749,0.364251,High Point
2,West,BYU,6,Gonzaga,3,0.208570,0.791430,Gonzaga
3,West,Miami (FL),7,Purdue,2,0.503860,0.496140,Miami (FL)


### A couple of those are especially interesting:

- High Point keeps advancing, which is a real sleeper-style signal
- Miami (FL) over Purdue is basically a toss-up, but the model leans Miami

### Make West round 2 winners table

In [650]:
west_round2_winners = west_round2[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

west_round2_winners["winner_seed"] = west_round2_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

west_round2_winners["winner_team_id"] = west_round2_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

west_round2_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,West,Arizona,1,1112,Utah St.,9,1429,Arizona,1,1112
1,West,High Point,12,1219,Arkansas,4,1116,High Point,12,1219
2,West,BYU,6,1140,Gonzaga,3,1211,Gonzaga,3,1211
3,West,Miami (FL),7,1274,Purdue,2,1345,Miami (FL),7,1274


### Build round 3

In [651]:
west_round3_rows = []

west_round3_rows.append({
    "region": "West",
    "team_a": west_round2_winners.iloc[0]["predicted_winner"],
    "seed_a": west_round2_winners.iloc[0]["winner_seed"],
    "team_a_id": west_round2_winners.iloc[0]["winner_team_id"],
    "team_b": west_round2_winners.iloc[1]["predicted_winner"],
    "seed_b": west_round2_winners.iloc[1]["winner_seed"],
    "team_b_id": west_round2_winners.iloc[1]["winner_team_id"],
})

west_round3_rows.append({
    "region": "West",
    "team_a": west_round2_winners.iloc[2]["predicted_winner"],
    "seed_a": west_round2_winners.iloc[2]["winner_seed"],
    "team_a_id": west_round2_winners.iloc[2]["winner_team_id"],
    "team_b": west_round2_winners.iloc[3]["predicted_winner"],
    "seed_b": west_round2_winners.iloc[3]["winner_seed"],
    "team_b_id": west_round2_winners.iloc[3]["winner_team_id"],
})

west_round3 = pd.DataFrame(west_round3_rows)
west_round3


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,West,Arizona,1,1112,High Point,12,1219
1,West,Gonzaga,3,1211,Miami (FL),7,1274


### Merge features and score west round 3

In [652]:
west_round3 = west_round3.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

west_round3 = west_round3.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

west_round3["win_pct_diff"] = west_round3["team_a_win_pct"] - west_round3["team_b_win_pct"]
west_round3["points_for_diff"] = west_round3["team_a_avg_points_for"] - west_round3["team_b_avg_points_for"]
west_round3["points_against_diff"] = west_round3["team_a_avg_points_against"] - west_round3["team_b_avg_points_against"]
west_round3["scoring_margin_diff"] = west_round3["team_a_avg_scoring_margin"] - west_round3["team_b_avg_scoring_margin"]

X_west_round3 = west_round3[feature_cols]

west_round3["team_a_win_prob"] = log_reg.predict_proba(X_west_round3)[:, 1]
west_round3["team_b_win_prob"] = 1 - west_round3["team_a_win_prob"]

west_round3["predicted_winner"] = west_round3.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

west_round3[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,West,Arizona,1,High Point,12,0.613853,0.386147,Arizona
1,West,Gonzaga,3,Miami (FL),7,0.716363,0.283637,Gonzaga


### Model Preicts
- Arizona over High Point
- Gonzaga over Miami (FL)

### Make round 3 winners table

In [653]:
west_round3_winners = west_round3[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

west_round3_winners["winner_seed"] = west_round3_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

west_round3_winners["winner_team_id"] = west_round3_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

west_round3_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,West,Arizona,1,1112,High Point,12,1219,Arizona,1,1112
1,West,Gonzaga,3,1211,Miami (FL),7,1274,Gonzaga,3,1211


### Build west final

In [654]:
west_final = pd.DataFrame([{
    "region": "West",
    "team_a": west_round3_winners.iloc[0]["predicted_winner"],
    "seed_a": west_round3_winners.iloc[0]["winner_seed"],
    "team_a_id": west_round3_winners.iloc[0]["winner_team_id"],
    "team_b": west_round3_winners.iloc[1]["predicted_winner"],
    "seed_b": west_round3_winners.iloc[1]["winner_seed"],
    "team_b_id": west_round3_winners.iloc[1]["winner_team_id"],
}])

west_final


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,West,Arizona,1,1112,Gonzaga,3,1211


### Merge features and score to west final

In [655]:
west_final = west_final.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

west_final = west_final.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

west_final["win_pct_diff"] = west_final["team_a_win_pct"] - west_final["team_b_win_pct"]
west_final["points_for_diff"] = west_final["team_a_avg_points_for"] - west_final["team_b_avg_points_for"]
west_final["points_against_diff"] = west_final["team_a_avg_points_against"] - west_final["team_b_avg_points_against"]
west_final["scoring_margin_diff"] = west_final["team_a_avg_scoring_margin"] - west_final["team_b_avg_scoring_margin"]

X_west_final = west_final[feature_cols]

west_final["team_a_win_prob"] = log_reg.predict_proba(X_west_final)[:, 1]
west_final["team_b_win_prob"] = 1 - west_final["team_a_win_prob"]

west_final["predicted_winner"] = west_final.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

west_final[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,West,Arizona,1,Gonzaga,3,0.454501,0.545499,Gonzaga


## West Region Result

The model now gives a full predicted path through the West region.

### Predicted West path
- Round 1 winners: Arizona, Utah St., High Point, Arkansas, BYU, Gonzaga, Miami (FL), Purdue
- Round 2 winners: Arizona, High Point, Gonzaga, Miami (FL)
- Round 3 winners: Arizona, Gonzaga
- West champion: Gonzaga

### Why this matters
This confirms that the same round-advancement pattern works in another fully resolved region. Winners can be advanced using `TeamID`, later-round matchup features can be rebuilt correctly, and the model can be reused at every round.


### The East results looked more intuitive to me, but in the West the model made at least one pick I would personally question, especially Gonzaga over Arizona. But the probability was still fairly close, so it is more of a model lean than a dominant prediction.

## East and West Advancement Heatmap

This chart shows how far each team advanced in the current baseline bracket path for the East and West regions.

At this stage, these are not Monte Carlo odds yet. They are the current round-by-round outcomes from the model's bracket path through the East and West.



In [656]:
import pandas as pd
import plotly.express as px

east_heatmap = pd.DataFrame([
    {"team": "Duke", "Round 2": 1, "Round 3": 1, "Final": 1, "Champion": 1},
    {"team": "TCU", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "St. John's", "Round 2": 1, "Round 3": 1, "Final": 0, "Champion": 0},
    {"team": "Kansas", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "Louisville", "Round 2": 1, "Round 3": 1, "Final": 1, "Champion": 0},
    {"team": "Michigan St.", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "UCLA", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "UConn", "Round 2": 1, "Round 3": 1, "Final": 0, "Champion": 0},
])

west_heatmap = pd.DataFrame([
    {"team": "Arizona", "Round 2": 1, "Round 3": 1, "Final": 1, "Champion": 0},
    {"team": "Utah St.", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "High Point", "Round 2": 1, "Round 3": 1, "Final": 0, "Champion": 0},
    {"team": "Arkansas", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "BYU", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
    {"team": "Gonzaga", "Round 2": 1, "Round 3": 1, "Final": 1, "Champion": 1},
    {"team": "Miami (FL)", "Round 2": 1, "Round 3": 1, "Final": 0, "Champion": 0},
    {"team": "Purdue", "Round 2": 1, "Round 3": 0, "Final": 0, "Champion": 0},
])

east_long = east_heatmap.melt(
    id_vars="team",
    var_name="Round",
    value_name="Reached"
)

west_long = west_heatmap.melt(
    id_vars="team",
    var_name="Round",
    value_name="Reached"
)

fig_east = px.density_heatmap(
    east_long,
    x="Round",
    y="team",
    z="Reached",
    text_auto=True,
    color_continuous_scale="Blues",
    title="East Region Advancement"
)

fig_east.update_layout(height=500)
fig_east.show()

fig_west = px.density_heatmap(
    west_long,
    x="Round",
    y="team",
    z="Reached",
    text_auto=True,
    color_continuous_scale="Greens",
    title="West Region Advancement"
)

fig_west.update_layout(height=500)
fig_west.show()


### Fixed the 2 Play-ins in csv and reran the notebook

In [657]:
bracket_play_in


,region,seed,team,team_clean,is_play_in,TeamID,TeamName


In [658]:
first_round_2026.shape


(32, 18)

### South round-1 predictions

### 

In [659]:
south_round1 = round1_predictions[round1_predictions["region"] == "South"].copy()
south_round1


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite
8,South,Florida,1,PVAMU,16,0.941828,0.058172,Florida,Florida
9,South,Clemson,8,Iowa,9,0.431105,0.568895,Iowa,Iowa
10,South,Vanderbilt,5,McNeese,12,0.600251,0.399749,Vanderbilt,Vanderbilt
11,South,Nebraska,4,Troy,13,0.780472,0.219528,Nebraska,Nebraska
12,South,North Carolina,6,VCU,11,0.463528,0.536472,VCU,VCU
13,South,Illinois,3,Penn,14,0.880012,0.119988,Illinois,Illinois
14,South,Saint Mary's,7,Texas A&M,10,0.592795,0.407205,Saint Mary's,Saint Mary's
15,South,Houston,2,Idaho,15,0.861132,0.138868,Houston,Houston


### Attach team_a_id

In [660]:
team_lookup = bracket_with_features[
    ["region", "seed", "team", "TeamID"]
].copy()

team_lookup.head()


,region,seed,team,TeamID
0,East,1,Duke,1181
1,East,16,Siena,1373
2,East,8,Ole Miss,1279
3,East,9,TCU,1395
4,East,5,St. John's,1385


In [661]:
team_lookup[team_lookup["region"] == "South"]


,region,seed,team,TeamID
16,South,1,Florida,1196
17,South,16,PVAMU,1341
18,South,8,Clemson,1155
19,South,9,Iowa,1234
20,South,5,Vanderbilt,1435
21,South,12,McNeese,1270
22,South,4,Nebraska,1304
23,South,13,Troy,1407
24,South,6,North Carolina,1314
25,South,11,VCU,1433


### attach team_a_id

In [662]:
south_round1_ids = south_round1.merge(
    team_lookup.rename(columns={"team": "team_a", "seed": "seed_a", "TeamID": "team_a_id"}),
    on=["region", "team_a", "seed_a"],
    how="left"
)


### attach team_b_id

In [663]:
south_round1_ids = south_round1_ids.merge(
    team_lookup.rename(columns={"team": "team_b", "seed": "seed_b", "TeamID": "team_b_id"}),
    on=["region", "team_b", "seed_b"],
    how="left"
)

south_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id,team_b_id
0,South,Florida,1,PVAMU,16,0.941828,0.058172,Florida,Florida,1196,1341
1,South,Clemson,8,Iowa,9,0.431105,0.568895,Iowa,Iowa,1155,1234
2,South,Vanderbilt,5,McNeese,12,0.600251,0.399749,Vanderbilt,Vanderbilt,1435,1270
3,South,Nebraska,4,Troy,13,0.780472,0.219528,Nebraska,Nebraska,1304,1407
4,South,North Carolina,6,VCU,11,0.463528,0.536472,VCU,VCU,1314,1433


### Build South round-1 winners table

In [664]:
south_round1_winners = south_round1_ids[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

south_round1_winners["winner_seed"] = south_round1_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

south_round1_winners["winner_team_id"] = south_round1_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

south_round1_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,South,Florida,1,1196,PVAMU,16,1341,Florida,1,1196
1,South,Clemson,8,1155,Iowa,9,1234,Iowa,9,1234
2,South,Vanderbilt,5,1435,McNeese,12,1270,Vanderbilt,5,1435
3,South,Nebraska,4,1304,Troy,13,1407,Nebraska,4,1304
4,South,North Carolina,6,1314,VCU,11,1433,VCU,11,1433
5,South,Illinois,3,1228,Penn,14,1335,Illinois,3,1228
6,South,Saint Mary's,7,1388,Texas A&M,10,1401,Saint Mary's,7,1388
7,South,Houston,2,1222,Idaho,15,1225,Houston,2,1222


### Build South round 2

In [665]:
first_round_2026[first_round_2026["region"] == "South"]


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff,team_a_win_prob,team_b_win_prob,predicted_winner,favorite_win_prob,underdog_pick,predicted_seed,expected_seed,is_upset_pick
8,South,1,Florida,16,PVAMU,-15,0.446950,10.368700,-10.586207,20.954907,0.941828,0.058172,Florida,0.941828,False,1,1,False
9,South,8,Clemson,9,Iowa,-1,0.034483,-0.965517,0.896552,-1.862069,0.431105,0.568895,Iowa,0.568895,True,9,8,True
10,South,5,Vanderbilt,12,McNeese,-7,-0.062808,10.217980,7.588670,2.629310,0.600251,0.399749,Vanderbilt,0.600251,False,5,5,False
11,South,4,Nebraska,13,Troy,-9,0.241379,0.241379,-9.655172,9.896552,0.780472,0.219528,Nebraska,0.780472,False,4,4,False
12,South,6,North Carolina,11,VCU,-5,0.034483,-2.448276,-1.655172,-0.793103,0.463528,0.536472,VCU,0.536472,True,11,6,True
13,South,3,Illinois,14,Penn,-11,0.198621,10.380690,-4.408276,14.788966,0.880012,0.119988,Illinois,0.880012,False,3,3,False
14,South,7,Saint Mary's,10,Texas A&M,-3,0.211494,-10.235632,-13.928736,3.693103,0.592795,0.407205,Saint Mary's,0.592795,False,7,7,False
15,South,2,Houston,15,Idaho,-13,0.346105,1.072797,-13.057471,14.130268,0.861132,0.138868,Houston,0.861132,False,2,2,False


In [666]:
bracket_with_features[
    (bracket_with_features["region"] == "South") & (bracket_with_features["seed"] == 16)
][["team", "team_clean", "TeamID", "TeamName"]]


,team,team_clean,TeamID,TeamName
17,PVAMU,Prairie View,1341,Prairie View


In [667]:
data["teams"][data["teams"]["TeamName"].str.contains("Prairie", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
240,1341,Prairie View,1985,2026


### Build South Round 2

In [668]:
south_round2_rows = []

for idx_a, idx_b in second_round_pairs:
    game_a = south_round1_winners.iloc[idx_a]
    game_b = south_round1_winners.iloc[idx_b]

    south_round2_rows.append({
        "region": "South",
        "team_a": game_a["predicted_winner"],
        "seed_a": game_a["winner_seed"],
        "team_a_id": game_a["winner_team_id"],
        "team_b": game_b["predicted_winner"],
        "seed_b": game_b["winner_seed"],
        "team_b_id": game_b["winner_team_id"],
    })

south_round2 = pd.DataFrame(south_round2_rows)
south_round2


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,South,Florida,1,1196,Iowa,9,1234
1,South,Vanderbilt,5,1435,Nebraska,4,1304
2,South,VCU,11,1433,Illinois,3,1228
3,South,Saint Mary's,7,1388,Houston,2,1222


### Score South Round 2

In [669]:
south_round2 = south_round2.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

south_round2 = south_round2.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)


In [670]:
south_round2["win_pct_diff"] = south_round2["team_a_win_pct"] - south_round2["team_b_win_pct"]
south_round2["points_for_diff"] = south_round2["team_a_avg_points_for"] - south_round2["team_b_avg_points_for"]
south_round2["points_against_diff"] = south_round2["team_a_avg_points_against"] - south_round2["team_b_avg_points_against"]
south_round2["scoring_margin_diff"] = south_round2["team_a_avg_scoring_margin"] - south_round2["team_b_avg_scoring_margin"]


In [671]:
X_south_round2 = south_round2[feature_cols]

south_round2["team_a_win_prob"] = log_reg.predict_proba(X_south_round2)[:, 1]
south_round2["team_b_win_prob"] = 1 - south_round2["team_a_win_prob"]

south_round2["predicted_winner"] = south_round2.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

south_round2[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,South,Florida,1,Iowa,9,0.672974,0.327026,Florida
1,South,Vanderbilt,5,Nebraska,4,0.487058,0.512942,Nebraska
2,South,VCU,11,Illinois,3,0.347989,0.652011,Illinois
3,South,Saint Mary's,7,Houston,2,0.417685,0.582315,Houston


### model now has:

- Florida over Iowa
- Nebraska over Vanderbilt
- Illinois over VCU
- Houston over Saint Mary's

### South round 3

In [672]:
south_round2_winners = south_round2[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

south_round2_winners["winner_seed"] = south_round2_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

south_round2_winners["winner_team_id"] = south_round2_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

south_round3_rows = []

south_round3_rows.append({
    "region": "South",
    "team_a": south_round2_winners.iloc[0]["predicted_winner"],
    "seed_a": south_round2_winners.iloc[0]["winner_seed"],
    "team_a_id": south_round2_winners.iloc[0]["winner_team_id"],
    "team_b": south_round2_winners.iloc[1]["predicted_winner"],
    "seed_b": south_round2_winners.iloc[1]["winner_seed"],
    "team_b_id": south_round2_winners.iloc[1]["winner_team_id"],
})

south_round3_rows.append({
    "region": "South",
    "team_a": south_round2_winners.iloc[2]["predicted_winner"],
    "seed_a": south_round2_winners.iloc[2]["winner_seed"],
    "team_a_id": south_round2_winners.iloc[2]["winner_team_id"],
    "team_b": south_round2_winners.iloc[3]["predicted_winner"],
    "seed_b": south_round2_winners.iloc[3]["winner_seed"],
    "team_b_id": south_round2_winners.iloc[3]["winner_team_id"],
})

south_round3 = pd.DataFrame(south_round3_rows)
south_round3


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,South,Florida,1,1196,Nebraska,4,1304
1,South,Illinois,3,1228,Houston,2,1222


In [673]:
south_round3 = south_round3.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

south_round3 = south_round3.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

south_round3["win_pct_diff"] = south_round3["team_a_win_pct"] - south_round3["team_b_win_pct"]
south_round3["points_for_diff"] = south_round3["team_a_avg_points_for"] - south_round3["team_b_avg_points_for"]
south_round3["points_against_diff"] = south_round3["team_a_avg_points_against"] - south_round3["team_b_avg_points_against"]
south_round3["scoring_margin_diff"] = south_round3["team_a_avg_scoring_margin"] - south_round3["team_b_avg_scoring_margin"]

X_south_round3 = south_round3[feature_cols]

south_round3["team_a_win_prob"] = log_reg.predict_proba(X_south_round3)[:, 1]
south_round3["team_b_win_prob"] = 1 - south_round3["team_a_win_prob"]

south_round3["predicted_winner"] = south_round3.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

south_round3[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,South,Florida,1,Nebraska,4,0.596203,0.403797,Florida
1,South,Illinois,3,Houston,2,0.503477,0.496523,Illinois


### Build and score the South Final

In [674]:
south_round3_winners = south_round3[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

south_round3_winners["winner_seed"] = south_round3_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

south_round3_winners["winner_team_id"] = south_round3_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

south_final = pd.DataFrame([{
    "region": "South",
    "team_a": south_round3_winners.iloc[0]["predicted_winner"],
    "seed_a": south_round3_winners.iloc[0]["winner_seed"],
    "team_a_id": south_round3_winners.iloc[0]["winner_team_id"],
    "team_b": south_round3_winners.iloc[1]["predicted_winner"],
    "seed_b": south_round3_winners.iloc[1]["winner_seed"],
    "team_b_id": south_round3_winners.iloc[1]["winner_team_id"],
}])

south_final = south_final.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

south_final = south_final.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

south_final["win_pct_diff"] = south_final["team_a_win_pct"] - south_final["team_b_win_pct"]
south_final["points_for_diff"] = south_final["team_a_avg_points_for"] - south_final["team_b_avg_points_for"]
south_final["points_against_diff"] = south_final["team_a_avg_points_against"] - south_final["team_b_avg_points_against"]
south_final["scoring_margin_diff"] = south_final["team_a_avg_scoring_margin"] - south_final["team_b_avg_scoring_margin"]

X_south_final = south_final[feature_cols]

south_final["team_a_win_prob"] = log_reg.predict_proba(X_south_final)[:, 1]
south_final["team_b_win_prob"] = 1 - south_final["team_a_win_prob"]

south_final["predicted_winner"] = south_final.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

south_final[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,South,Florida,1,Illinois,3,0.520184,0.479816,Florida


## South Region Result

The model now gives a full predicted path through the South region.

### Predicted South path
- Round 1 winners: Florida, Iowa, Vanderbilt, Nebraska, VCU, Illinois, Saint Mary's, Houston
- Round 2 winners: Florida, Nebraska, Illinois, Houston
- Round 3 winners: Florida, Illinois
- South champion: Florida

### Why this matters
This confirms that the round-advancement pattern also works after a play-in winner is inserted into the bracket. Winners can still be advanced using `TeamID`, later-round matchup features can still be rebuilt correctly, and the baseline model can still be reused at each round.

The South region also includes several close model decisions, which is a good reminder that these are still probabilities and model leans, not guaranteed outcomes.


### Midwest round 1 Predicitons

In [675]:
midwest_round1 = round1_predictions[round1_predictions["region"] == "Midwest"].copy()
midwest_round1


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite
24,Midwest,Michigan,1,Howard,16,0.870002,0.129998,Michigan,Michigan
25,Midwest,Georgia,8,Saint Louis,9,0.286159,0.713841,Saint Louis,Saint Louis
26,Midwest,Texas Tech,5,Akron,12,0.406010,0.593990,Akron,Akron
27,Midwest,Alabama,4,Hofstra,13,0.618832,0.381168,Alabama,Alabama
28,Midwest,Tennessee,6,Miami OH,11,0.449202,0.550798,Miami OH,Miami OH
29,Midwest,Virginia,3,Wright St.,14,0.777805,0.222195,Virginia,Virginia
30,Midwest,Kentucky,7,Santa Clara,10,0.424956,0.575044,Santa Clara,Santa Clara
31,Midwest,Iowa St.,2,Tennessee St.,15,0.863707,0.136293,Iowa St.,Iowa St.


### attach team_a_id 

In [676]:
midwest_round1_ids = midwest_round1.merge(
    team_lookup.rename(columns={"team": "team_a", "seed": "seed_a", "TeamID": "team_a_id"}),
    on=["region", "team_a", "seed_a"],
    how="left"
)


### attach team_b_id

In [677]:
midwest_round1_ids = midwest_round1_ids.merge(
    team_lookup.rename(columns={"team": "team_b", "seed": "seed_b", "TeamID": "team_b_id"}),
    on=["region", "team_b", "seed_b"],
    how="left"
)

midwest_round1_ids.head()


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner,favorite,team_a_id,team_b_id
0,Midwest,Michigan,1,Howard,16,0.870002,0.129998,Michigan,Michigan,1276,1224
1,Midwest,Georgia,8,Saint Louis,9,0.286159,0.713841,Saint Louis,Saint Louis,1208,1387
2,Midwest,Texas Tech,5,Akron,12,0.406010,0.593990,Akron,Akron,1403,1103
3,Midwest,Alabama,4,Hofstra,13,0.618832,0.381168,Alabama,Alabama,1104,1220
4,Midwest,Tennessee,6,Miami OH,11,0.449202,0.550798,Miami OH,Miami OH,1397,1275


### Build midwest round 1 winners table

In [678]:
midwest_round1_winners = midwest_round1_ids[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

midwest_round1_winners["winner_seed"] = midwest_round1_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

midwest_round1_winners["winner_team_id"] = midwest_round1_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

midwest_round1_winners


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id,predicted_winner,winner_seed,winner_team_id
0,Midwest,Michigan,1,1276,Howard,16,1224,Michigan,1,1276
1,Midwest,Georgia,8,1208,Saint Louis,9,1387,Saint Louis,9,1387
2,Midwest,Texas Tech,5,1403,Akron,12,1103,Akron,12,1103
3,Midwest,Alabama,4,1104,Hofstra,13,1220,Alabama,4,1104
4,Midwest,Tennessee,6,1397,Miami OH,11,1275,Miami OH,11,1275
5,Midwest,Virginia,3,1438,Wright St.,14,1460,Virginia,3,1438
6,Midwest,Kentucky,7,1246,Santa Clara,10,1365,Santa Clara,10,1365
7,Midwest,Iowa St.,2,1235,Tennessee St.,15,1398,Iowa St.,2,1235


### Build and score Midwest round 2

In [679]:
midwest_round2_rows = []

for idx_a, idx_b in second_round_pairs:
    game_a = midwest_round1_winners.iloc[idx_a]
    game_b = midwest_round1_winners.iloc[idx_b]

    midwest_round2_rows.append({
        "region": "Midwest",
        "team_a": game_a["predicted_winner"],
        "seed_a": game_a["winner_seed"],
        "team_a_id": game_a["winner_team_id"],
        "team_b": game_b["predicted_winner"],
        "seed_b": game_b["winner_seed"],
        "team_b_id": game_b["winner_team_id"],
    })

midwest_round2 = pd.DataFrame(midwest_round2_rows)
midwest_round2


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,Midwest,Michigan,1,1276,Saint Louis,9,1387
1,Midwest,Akron,12,1103,Alabama,4,1104
2,Midwest,Miami OH,11,1275,Virginia,3,1438
3,Midwest,Santa Clara,10,1365,Iowa St.,2,1235


In [680]:
midwest_round2 = midwest_round2.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

midwest_round2 = midwest_round2.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

midwest_round2["win_pct_diff"] = midwest_round2["team_a_win_pct"] - midwest_round2["team_b_win_pct"]
midwest_round2["points_for_diff"] = midwest_round2["team_a_avg_points_for"] - midwest_round2["team_b_avg_points_for"]
midwest_round2["points_against_diff"] = midwest_round2["team_a_avg_points_against"] - midwest_round2["team_b_avg_points_against"]
midwest_round2["scoring_margin_diff"] = midwest_round2["team_a_avg_scoring_margin"] - midwest_round2["team_b_avg_scoring_margin"]

X_midwest_round2 = midwest_round2[feature_cols]

midwest_round2["team_a_win_prob"] = log_reg.predict_proba(X_midwest_round2)[:, 1]
midwest_round2["team_b_win_prob"] = 1 - midwest_round2["team_a_win_prob"]

midwest_round2["predicted_winner"] = midwest_round2.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

midwest_round2[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,Midwest,Michigan,1,Saint Louis,9,0.558493,0.441507,Michigan
1,Midwest,Akron,12,Alabama,4,0.591256,0.408744,Akron
2,Midwest,Miami OH,11,Virginia,3,0.466624,0.533376,Virginia
3,Midwest,Santa Clara,10,Iowa St.,2,0.303497,0.696503,Iowa St.


### Build Midwest round 3

In [681]:
midwest_round2_winners = midwest_round2[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

midwest_round2_winners["winner_seed"] = midwest_round2_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

midwest_round2_winners["winner_team_id"] = midwest_round2_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

midwest_round3_rows = []

midwest_round3_rows.append({
    "region": "Midwest",
    "team_a": midwest_round2_winners.iloc[0]["predicted_winner"],
    "seed_a": midwest_round2_winners.iloc[0]["winner_seed"],
    "team_a_id": midwest_round2_winners.iloc[0]["winner_team_id"],
    "team_b": midwest_round2_winners.iloc[1]["predicted_winner"],
    "seed_b": midwest_round2_winners.iloc[1]["winner_seed"],
    "team_b_id": midwest_round2_winners.iloc[1]["winner_team_id"],
})

midwest_round3_rows.append({
    "region": "Midwest",
    "team_a": midwest_round2_winners.iloc[2]["predicted_winner"],
    "seed_a": midwest_round2_winners.iloc[2]["winner_seed"],
    "team_a_id": midwest_round2_winners.iloc[2]["winner_team_id"],
    "team_b": midwest_round2_winners.iloc[3]["predicted_winner"],
    "seed_b": midwest_round2_winners.iloc[3]["winner_seed"],
    "team_b_id": midwest_round2_winners.iloc[3]["winner_team_id"],
})

midwest_round3 = pd.DataFrame(midwest_round3_rows)
midwest_round3


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,Midwest,Michigan,1,1276,Akron,12,1103
1,Midwest,Virginia,3,1438,Iowa St.,2,1235


### Score it

In [682]:
midwest_round3 = midwest_round3.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

midwest_round3 = midwest_round3.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

midwest_round3["win_pct_diff"] = midwest_round3["team_a_win_pct"] - midwest_round3["team_b_win_pct"]
midwest_round3["points_for_diff"] = midwest_round3["team_a_avg_points_for"] - midwest_round3["team_b_avg_points_for"]
midwest_round3["points_against_diff"] = midwest_round3["team_a_avg_points_against"] - midwest_round3["team_b_avg_points_against"]
midwest_round3["scoring_margin_diff"] = midwest_round3["team_a_avg_scoring_margin"] - midwest_round3["team_b_avg_scoring_margin"]

X_midwest_round3 = midwest_round3[feature_cols]

midwest_round3["team_a_win_prob"] = log_reg.predict_proba(X_midwest_round3)[:, 1]
midwest_round3["team_b_win_prob"] = 1 - midwest_round3["team_a_win_prob"]

midwest_round3["predicted_winner"] = midwest_round3.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

midwest_round3[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,Midwest,Michigan,1,Akron,12,0.737095,0.262905,Michigan
1,Midwest,Virginia,3,Iowa St.,2,0.369311,0.630689,Iowa St.


### Build and score the Midwest final

In [683]:
midwest_round3_winners = midwest_round3[
    ["region", "team_a", "seed_a", "team_a_id", "team_b", "seed_b", "team_b_id", "predicted_winner"]
].copy()

midwest_round3_winners["winner_seed"] = midwest_round3_winners.apply(
    lambda row: row["seed_a"] if row["predicted_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

midwest_round3_winners["winner_team_id"] = midwest_round3_winners.apply(
    lambda row: row["team_a_id"] if row["predicted_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

midwest_final = pd.DataFrame([{
    "region": "Midwest",
    "team_a": midwest_round3_winners.iloc[0]["predicted_winner"],
    "seed_a": midwest_round3_winners.iloc[0]["winner_seed"],
    "team_a_id": midwest_round3_winners.iloc[0]["winner_team_id"],
    "team_b": midwest_round3_winners.iloc[1]["predicted_winner"],
    "seed_b": midwest_round3_winners.iloc[1]["winner_seed"],
    "team_b_id": midwest_round3_winners.iloc[1]["winner_team_id"],
}])

midwest_final = midwest_final.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

midwest_final = midwest_final.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

midwest_final["win_pct_diff"] = midwest_final["team_a_win_pct"] - midwest_final["team_b_win_pct"]
midwest_final["points_for_diff"] = midwest_final["team_a_avg_points_for"] - midwest_final["team_b_avg_points_for"]
midwest_final["points_against_diff"] = midwest_final["team_a_avg_points_against"] - midwest_final["team_b_avg_points_against"]
midwest_final["scoring_margin_diff"] = midwest_final["team_a_avg_scoring_margin"] - midwest_final["team_b_avg_scoring_margin"]

X_midwest_final = midwest_final[feature_cols]

midwest_final["team_a_win_prob"] = log_reg.predict_proba(X_midwest_final)[:, 1]
midwest_final["team_b_win_prob"] = 1 - midwest_final["team_a_win_prob"]

midwest_final["predicted_winner"] = midwest_final.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

midwest_final[
    ["region", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,region,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,Midwest,Michigan,1,Iowa St.,2,0.610413,0.389587,Michigan


## Midwest Region Result

The model now gives a full predicted path through the Midwest region.

### Predicted Midwest path
- Round 1 winners: Michigan, Saint Louis, Akron, Alabama, Miami OH, Virginia, Santa Clara, Iowa St.
- Round 2 winners: Michigan, Akron, Virginia, Iowa St.
- Round 3 winners: Michigan, Iowa St.
- Midwest champion: Michigan

### Why this matters
This completes the fourth and final region, which means the bracket is now advanced far enough to build the Final Four and national championship game. At this point, the regional advancement logic is working across the full tournament field.


## Regional Champions Summary

At this point, all four regions have been advanced through their regional finals using the baseline model and the round-by-round `TeamID` advancement logic.

### Current Final Four teams
- East champion: Duke
- West champion: Gonzaga
- South champion: Florida
- Midwest champion: Michigan

This means the bracket is now ready for the Final Four matchups and the national championship game.


### Building Final Four matchup table

In [684]:
final_four = pd.DataFrame([
    {
        "round": "Final Four",
        "team_a": south_final.iloc[0]["predicted_winner"],
        "seed_a": south_final.iloc[0]["seed_a"] if south_final.iloc[0]["predicted_winner"] == south_final.iloc[0]["team_a"] else south_final.iloc[0]["seed_b"],
        "team_a_id": south_final.iloc[0]["team_a_id"] if south_final.iloc[0]["predicted_winner"] == south_final.iloc[0]["team_a"] else south_final.iloc[0]["team_b_id"],
        "team_b": west_final.iloc[0]["predicted_winner"],
        "seed_b": west_final.iloc[0]["seed_a"] if west_final.iloc[0]["predicted_winner"] == west_final.iloc[0]["team_a"] else west_final.iloc[0]["seed_b"],
        "team_b_id": west_final.iloc[0]["team_a_id"] if west_final.iloc[0]["predicted_winner"] == west_final.iloc[0]["team_a"] else west_final.iloc[0]["team_b_id"],
    },
    {
        "round": "Final Four",
        "team_a": east_final.iloc[0]["predicted_winner"],
        "seed_a": east_final.iloc[0]["seed_a"] if east_final.iloc[0]["predicted_winner"] == east_final.iloc[0]["team_a"] else east_final.iloc[0]["seed_b"],
        "team_a_id": east_final.iloc[0]["team_a_id"] if east_final.iloc[0]["predicted_winner"] == east_final.iloc[0]["team_a"] else east_final.iloc[0]["team_b_id"],
        "team_b": midwest_final.iloc[0]["predicted_winner"],
        "seed_b": midwest_final.iloc[0]["seed_a"] if midwest_final.iloc[0]["predicted_winner"] == midwest_final.iloc[0]["team_a"] else midwest_final.iloc[0]["seed_b"],
        "team_b_id": midwest_final.iloc[0]["team_a_id"] if midwest_final.iloc[0]["predicted_winner"] == midwest_final.iloc[0]["team_a"] else midwest_final.iloc[0]["team_b_id"],
    },
])

final_four


,round,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,Final Four,Florida,1,1196,Gonzaga,3,1211
1,Final Four,Duke,1,1181,Michigan,1,1276


### Merge features and score Final Four

In [685]:
final_four = final_four.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

final_four = final_four.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

final_four["win_pct_diff"] = final_four["team_a_win_pct"] - final_four["team_b_win_pct"]
final_four["points_for_diff"] = final_four["team_a_avg_points_for"] - final_four["team_b_avg_points_for"]
final_four["points_against_diff"] = final_four["team_a_avg_points_against"] - final_four["team_b_avg_points_against"]
final_four["scoring_margin_diff"] = final_four["team_a_avg_scoring_margin"] - final_four["team_b_avg_scoring_margin"]

X_final_four = final_four[feature_cols]

final_four["team_a_win_prob"] = log_reg.predict_proba(X_final_four)[:, 1]
final_four["team_b_win_prob"] = 1 - final_four["team_a_win_prob"]

final_four["predicted_winner"] = final_four.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

final_four[
    ["round", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,round,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,Final Four,Florida,1,Gonzaga,3,0.370737,0.629263,Gonzaga
1,Final Four,Duke,1,Michigan,1,0.487534,0.512466,Michigan


### model now has:

- Gonzaga over Florida
- Michigan over Duke

### Build the championship game

In [686]:
championship = pd.DataFrame([{
    "round": "Championship",
    "team_a": final_four.iloc[0]["predicted_winner"],
    "seed_a": final_four.iloc[0]["seed_a"] if final_four.iloc[0]["predicted_winner"] == final_four.iloc[0]["team_a"] else final_four.iloc[0]["seed_b"],
    "team_a_id": final_four.iloc[0]["team_a_id"] if final_four.iloc[0]["predicted_winner"] == final_four.iloc[0]["team_a"] else final_four.iloc[0]["team_b_id"],
    "team_b": final_four.iloc[1]["predicted_winner"],
    "seed_b": final_four.iloc[1]["seed_a"] if final_four.iloc[1]["predicted_winner"] == final_four.iloc[1]["team_a"] else final_four.iloc[1]["seed_b"],
    "team_b_id": final_four.iloc[1]["team_a_id"] if final_four.iloc[1]["predicted_winner"] == final_four.iloc[1]["team_a"] else final_four.iloc[1]["team_b_id"],
}])

championship


,round,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,Championship,Gonzaga,3,1211,Michigan,1,1276


### Score it

In [687]:
championship = championship.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

championship = championship.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

championship["win_pct_diff"] = championship["team_a_win_pct"] - championship["team_b_win_pct"]
championship["points_for_diff"] = championship["team_a_avg_points_for"] - championship["team_b_avg_points_for"]
championship["points_against_diff"] = championship["team_a_avg_points_against"] - championship["team_b_avg_points_against"]
championship["scoring_margin_diff"] = championship["team_a_avg_scoring_margin"] - championship["team_b_avg_scoring_margin"]

X_championship = championship[feature_cols]

championship["team_a_win_prob"] = log_reg.predict_proba(X_championship)[:, 1]
championship["team_b_win_prob"] = 1 - championship["team_a_win_prob"]

championship["predicted_winner"] = championship.apply(
    lambda row: row["team_a"] if row["team_a_win_prob"] >= 0.5 else row["team_b"],
    axis=1
)

championship[
    ["round", "team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "predicted_winner"]
]


,round,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,predicted_winner
0,Championship,Gonzaga,3,Michigan,1,0.472755,0.527245,Michigan


## Championship Result

The baseline model now gives a full tournament path through all four regions, the Final Four, and the championship game.

### Final Four results
- Gonzaga over Florida
- Michigan over Duke

### National Championship result
- Michigan over Gonzaga

### Predicted national champion
- Michigan

### Why this matters
This is the first full end-to-end bracket run using the current baseline model. At this point, the project is no longer limited to first-round predictions. The full tournament structure is now working from regional rounds through the national title game.


## What Monte Carlo Simulation Will Add

Up to this point, the project has produced one full baseline bracket path by advancing the team with the higher predicted win probability in each game.

That gives a useful first full-bracket result, but it is still deterministic. In real tournament play, a team with a 55% win probability does not always win. Monte Carlo simulation is the next step because it allows the bracket to behave more like a real tournament with uncertainty, randomness, and upset potential.

### What Monte Carlo will do
Instead of always advancing the higher-probability team, Monte Carlo simulation will:
- use the model's predicted win probability for each matchup
- randomly sample a winner based on that probability
- advance that winner to the next round
- repeat that process through the full bracket
- run the entire tournament many times

### Why this matters
This will turn the project from one single predicted bracket into a distribution of possible outcomes. That makes it possible to estimate:
- championship odds
- Final Four odds
- deep-run probabilities
- common upset paths
- sleeper teams that advance more often than expected

### Will Monte Carlo pick different winners?
Yes, it definitely can.

The current bracket path always picks the team with the higher probability, but Monte Carlo will sometimes pick the other team when the probability is still realistic. For example, if one team has a 53% chance to win, the other team still wins 47% of the time in simulation. That means some tournament runs will look different from the current bracket, especially in close games.

So the current bracket should be viewed as one baseline path, while Monte Carlo will show how often other paths happen and which outcomes appear most often overall.


## Monte Carlo Simulation

This section starts a simple Monte Carlo version of the bracket.

Instead of always advancing the higher-probability team, the simulation will randomly choose winners based on the model's predicted win probabilities. This makes the bracket behave more like a real tournament where favorites can still lose and close games can flip in different runs.


In [688]:
import numpy as np


### Quick simulated winner column for round-1 table

In [689]:
round1_sim = round1_predictions.copy()

round1_sim["simulated_winner"] = round1_sim.apply(
    lambda row: row["team_a"] if np.random.random() < row["team_a_win_prob"] else row["team_b"],
    axis=1
)

round1_sim[
    ["region", "team_a", "team_b", "team_a_win_prob", "team_b_win_prob", "simulated_winner"]
]


,region,team_a,team_b,team_a_win_prob,team_b_win_prob,simulated_winner
0,East,Duke,Siena,0.896083,0.103917,Duke
1,East,Ole Miss,TCU,0.282931,0.717069,Ole Miss
2,East,St. John's,Northern Iowa,0.687494,0.312506,St. John's
3,East,Kansas,Cal Baptist,0.609764,0.390236,Cal Baptist
4,East,Louisville,South Florida,0.619817,0.380183,Louisville
5,East,Michigan St.,North Dakota St.,0.630652,0.369348,Michigan St.
6,East,UCLA,UCF,0.546426,0.453574,UCF
7,East,UConn,Furman,0.812610,0.187390,Furman
8,South,Florida,PVAMU,0.941828,0.058172,Florida
9,South,Clemson,Iowa,0.431105,0.568895,Iowa


### Simulate East round 1

In [690]:
east_round1_sim = east_round1_ids.copy()

east_round1_sim["simulated_winner"] = east_round1_sim.apply(
    lambda row: row["team_a"] if np.random.random() < row["team_a_win_prob"] else row["team_b"],
    axis=1
)

east_round1_sim["winner_seed"] = east_round1_sim.apply(
    lambda row: row["seed_a"] if row["simulated_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round1_sim["winner_team_id"] = east_round1_sim.apply(
    lambda row: row["team_a_id"] if row["simulated_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round1_sim[
    ["team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "simulated_winner"]
]


,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,simulated_winner
0,Duke,1,Siena,16,0.896083,0.103917,Duke
1,Ole Miss,8,TCU,9,0.282931,0.717069,TCU
2,St. John's,5,Northern Iowa,12,0.687494,0.312506,Northern Iowa
3,Kansas,4,Cal Baptist,13,0.609764,0.390236,Kansas
4,Louisville,6,South Florida,11,0.619817,0.380183,Louisville
5,Michigan St.,3,North Dakota St.,14,0.630652,0.369348,North Dakota St.
6,UCLA,7,UCF,10,0.546426,0.453574,UCF
7,UConn,2,Furman,15,0.812610,0.187390,UConn


### Build simulated East round 2

In [691]:
east_round2_sim_rows = []

for idx_a, idx_b in second_round_pairs:
    game_a = east_round1_sim.iloc[idx_a]
    game_b = east_round1_sim.iloc[idx_b]

    east_round2_sim_rows.append({
        "region": "East",
        "team_a": game_a["simulated_winner"],
        "seed_a": game_a["winner_seed"],
        "team_a_id": game_a["winner_team_id"],
        "team_b": game_b["simulated_winner"],
        "seed_b": game_b["winner_seed"],
        "team_b_id": game_b["winner_team_id"],
    })

east_round2_sim = pd.DataFrame(east_round2_sim_rows)
east_round2_sim


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,East,Duke,1,1181,TCU,9,1395
1,East,Northern Iowa,12,1320,Kansas,4,1242
2,East,Louisville,6,1257,North Dakota St.,14,1295
3,East,UCF,10,1416,UConn,2,1163


### Score these simulated round 2 games

In [692]:
east_round2_sim = east_round2_sim.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

east_round2_sim = east_round2_sim.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

east_round2_sim["win_pct_diff"] = east_round2_sim["team_a_win_pct"] - east_round2_sim["team_b_win_pct"]
east_round2_sim["points_for_diff"] = east_round2_sim["team_a_avg_points_for"] - east_round2_sim["team_b_avg_points_for"]
east_round2_sim["points_against_diff"] = east_round2_sim["team_a_avg_points_against"] - east_round2_sim["team_b_avg_points_against"]
east_round2_sim["scoring_margin_diff"] = east_round2_sim["team_a_avg_scoring_margin"] - east_round2_sim["team_b_avg_scoring_margin"]

X_east_round2_sim = east_round2_sim[feature_cols]

east_round2_sim["team_a_win_prob"] = log_reg.predict_proba(X_east_round2_sim)[:, 1]
east_round2_sim["team_b_win_prob"] = 1 - east_round2_sim["team_a_win_prob"]


### Sample simulated round 2 winners

In [693]:
east_round2_sim["simulated_winner"] = east_round2_sim.apply(
    lambda row: row["team_a"] if np.random.random() < row["team_a_win_prob"] else row["team_b"],
    axis=1
)

east_round2_sim["winner_seed"] = east_round2_sim.apply(
    lambda row: row["seed_a"] if row["simulated_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round2_sim["winner_team_id"] = east_round2_sim.apply(
    lambda row: row["team_a_id"] if row["simulated_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round2_sim[
    ["team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "simulated_winner"]
]


,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,simulated_winner
0,Duke,1,TCU,9,0.861756,0.138244,Duke
1,Northern Iowa,12,Kansas,4,0.447637,0.552363,Kansas
2,Louisville,6,North Dakota St.,14,0.697616,0.302384,Louisville
3,UCF,10,UConn,2,0.233530,0.766470,UConn


### Build simulated East round 3

In [694]:
east_round3_sim_rows = []

east_round3_sim_rows.append({
    "region": "East",
    "team_a": east_round2_sim.iloc[0]["simulated_winner"],
    "seed_a": east_round2_sim.iloc[0]["winner_seed"],
    "team_a_id": east_round2_sim.iloc[0]["winner_team_id"],
    "team_b": east_round2_sim.iloc[1]["simulated_winner"],
    "seed_b": east_round2_sim.iloc[1]["winner_seed"],
    "team_b_id": east_round2_sim.iloc[1]["winner_team_id"],
})

east_round3_sim_rows.append({
    "region": "East",
    "team_a": east_round2_sim.iloc[2]["simulated_winner"],
    "seed_a": east_round2_sim.iloc[2]["winner_seed"],
    "team_a_id": east_round2_sim.iloc[2]["winner_team_id"],
    "team_b": east_round2_sim.iloc[3]["simulated_winner"],
    "seed_b": east_round2_sim.iloc[3]["winner_seed"],
    "team_b_id": east_round2_sim.iloc[3]["winner_team_id"],
})

east_round3_sim = pd.DataFrame(east_round3_sim_rows)
east_round3_sim


,region,team_a,seed_a,team_a_id,team_b,seed_b,team_b_id
0,East,Duke,1,1181,Kansas,4,1242
1,East,Louisville,6,1257,UConn,2,1163


### Score and simulate round 3

In [695]:
east_round3_sim = east_round3_sim.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

east_round3_sim = east_round3_sim.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

east_round3_sim["win_pct_diff"] = east_round3_sim["team_a_win_pct"] - east_round3_sim["team_b_win_pct"]
east_round3_sim["points_for_diff"] = east_round3_sim["team_a_avg_points_for"] - east_round3_sim["team_b_avg_points_for"]
east_round3_sim["points_against_diff"] = east_round3_sim["team_a_avg_points_against"] - east_round3_sim["team_b_avg_points_against"]
east_round3_sim["scoring_margin_diff"] = east_round3_sim["team_a_avg_scoring_margin"] - east_round3_sim["team_b_avg_scoring_margin"]

X_east_round3_sim = east_round3_sim[feature_cols]

east_round3_sim["team_a_win_prob"] = log_reg.predict_proba(X_east_round3_sim)[:, 1]
east_round3_sim["team_b_win_prob"] = 1 - east_round3_sim["team_a_win_prob"]

east_round3_sim["simulated_winner"] = east_round3_sim.apply(
    lambda row: row["team_a"] if np.random.random() < row["team_a_win_prob"] else row["team_b"],
    axis=1
)

east_round3_sim["winner_seed"] = east_round3_sim.apply(
    lambda row: row["seed_a"] if row["simulated_winner"] == row["team_a"] else row["seed_b"],
    axis=1
)

east_round3_sim["winner_team_id"] = east_round3_sim.apply(
    lambda row: row["team_a_id"] if row["simulated_winner"] == row["team_a"] else row["team_b_id"],
    axis=1
)

east_round3_sim[
    ["team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "simulated_winner"]
]


,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,simulated_winner
0,Duke,1,Kansas,4,0.848681,0.151319,Duke
1,Louisville,6,UConn,2,0.519357,0.480643,Louisville


### Simulate the East final

In [696]:
east_final_sim = pd.DataFrame([{
    "region": "East",
    "team_a": east_round3_sim.iloc[0]["simulated_winner"],
    "seed_a": east_round3_sim.iloc[0]["winner_seed"],
    "team_a_id": east_round3_sim.iloc[0]["winner_team_id"],
    "team_b": east_round3_sim.iloc[1]["simulated_winner"],
    "seed_b": east_round3_sim.iloc[1]["winner_seed"],
    "team_b_id": east_round3_sim.iloc[1]["winner_team_id"],
}])

east_final_sim = east_final_sim.merge(
    features_2026.add_prefix("team_a_"),
    left_on="team_a_id",
    right_on="team_a_TeamID",
    how="left"
)

east_final_sim = east_final_sim.merge(
    features_2026.add_prefix("team_b_"),
    left_on="team_b_id",
    right_on="team_b_TeamID",
    how="left"
)

east_final_sim["win_pct_diff"] = east_final_sim["team_a_win_pct"] - east_final_sim["team_b_win_pct"]
east_final_sim["points_for_diff"] = east_final_sim["team_a_avg_points_for"] - east_final_sim["team_b_avg_points_for"]
east_final_sim["points_against_diff"] = east_final_sim["team_a_avg_points_against"] - east_final_sim["team_b_avg_points_against"]
east_final_sim["scoring_margin_diff"] = east_final_sim["team_a_avg_scoring_margin"] - east_final_sim["team_b_avg_scoring_margin"]

X_east_final_sim = east_final_sim[feature_cols]

east_final_sim["team_a_win_prob"] = log_reg.predict_proba(X_east_final_sim)[:, 1]
east_final_sim["team_b_win_prob"] = 1 - east_final_sim["team_a_win_prob"]

east_final_sim["simulated_winner"] = east_final_sim.apply(
    lambda row: row["team_a"] if np.random.random() < row["team_a_win_prob"] else row["team_b"],
    axis=1
)

east_final_sim[
    ["team_a", "seed_a", "team_b", "seed_b", "team_a_win_prob", "team_b_win_prob", "simulated_winner"]
]


,team_a,seed_a,team_b,seed_b,team_a_win_prob,team_b_win_prob,simulated_winner
0,Duke,1,Louisville,6,0.692216,0.307784,Duke
